# 第 84 课：声明式控制器的漂移检测循环、幂等调谐与安全降级

上一课把发布配置、实际权重、WAL 和双副本 KV 恢复连接起来。这一课再增加一个**受限的声明式调谐器**，让它反复比较“期望”与“实际”，并决定什么时候自动修、什么时候停流量、什么时候升级人工处理。

本课完成的是**可以运行的教学控制循环**：真实 PyTorch Transformer、Q/V LoRA、真实 CPU KV 张量、双副本本地 WAL、配置代际与文件审计。控制者、租约、TTL、退避均在**单进程、显式逻辑 tick**中模拟，不能直接替代 Kubernetes controller-manager、外部 IAM、etcd 事务或多节点共识。

**核心安全边界：只有单个副本的 `telemetry_sample_percent` 一处漂移允许自动修复；路由、租户权限、容量、模型身份、WAL 或损坏配置都必须阻断新请求并交人工。**

## 0. 使用方式与验收标准

已有 PyTorch 环境即可，默认 CPU，不下载预训练模型、不调用云服务。选择 Jupyter 的 Python 内核，从头到尾运行。配套脚本独立运行，Notebook 也包含全部源码，不依赖前面章节。

整体验收关注六件事：

1. **幂等**：健康状态重复调谐不重复发布或覆盖；同一个逻辑 tick 的重复事件没有副作用。
2. **安全范围**：宽限期内不修；未知字段、权限、模型和 WAL 漂移不静默覆盖。
3. **有限失败处理**：2、4、8 tick 退避，失败到上限进入人工隔离。
4. **控制者围栏**：过期 lease、旧 term 与旧 epoch 不能执行修复。
5. **冷恢复**：修复成功但状态尚未写盘时崩溃，下次先读真实状态，不重放已成功的命令。
6. **真实推理**：增量 KV logits 与完整上下文一致；终态物理 KV 页归零。

默认执行完整演练。练习检查默认 `False`，练习本身需要亲手补全。所有时间单位是逻辑 tick，不能解释成 GPU 毫秒或真实墙钟时延。

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""第84课：声明式控制器、幂等调谐与安全降级。

完整离线、CPU PyTorch：真实自回归Transformer、Q/V LoRA与物理KV张量。
承接双副本WAL与物理KV，演示声明式配置、CAS发布、漂移阻断与恢复后验收。
仅为单进程模拟的双实例；不提供真实集群共识、远端存储、网络或IAM。
所有RTO/RPO单位是显式的教学逻辑tick；不宣称GPU毫秒性能。
"""
from __future__ import annotations

# %% 01 环境与适用边界
import argparse
import copy
import hashlib
import json
import math
import os
import platform
import random
import re
import tempfile
import time
from collections import Counter
from dataclasses import dataclass, asdict, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable

try:
    import torch
    from torch import nn
    from torch.nn import functional as F
except ImportError as exc:
    raise SystemExit('需要已有PyTorch环境；不会自动安装或下载。') from exc

VERSION = 'lesson84-declarative-controller-v1'
SOURCES = {
    'k8s_server_side_apply': 'https://kubernetes.io/docs/reference/using-api/server-side-apply/',
    'nist_config_management': 'https://csrc.nist.gov/pubs/sp/800/128/upd1/final',
    'sre_canary': 'https://sre.google/workbook/canarying-releases/',
    'sre_change': 'https://sre.google/sre-book/reliable-product-launches/',
    'torch_random': 'https://docs.pytorch.org/docs/stable/notes/randomness.html',
    'k8s_controller': 'https://kubernetes.io/docs/concepts/architecture/controller/',
    'k8s_ssa': 'https://kubernetes.io/docs/reference/using-api/server-side-apply/',
    'operator_errors': 'https://www.redhat.com/en/blog/kubernetes-operators-best-practices',
}

class LessonError(ValueError):
    """拒绝含糊的token边界、停止状态、报告格式或资源预算。"""


def require(ok: bool, msg: str) -> None:
    if not bool(ok):
        raise LessonError(msg)


def canonical(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)


def digest(value: Any) -> str:
    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()


def write_json(path: Path, value: Any) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True,
                               allow_nan=False) + "\n", encoding="utf-8")


def bounded_str(value: Any, length: int, label: str) -> str:
    require(type(value) is str and len(value.encode("utf-8")) <= length,
            label + "必须是有界字符串")
    return value


def seed_all(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def state_hash(state: dict[str,torch.Tensor]) -> str:
    h=hashlib.sha256()
    for key,tensor in sorted(state.items()):
        t=tensor.detach().cpu().contiguous()
        h.update(canonical([key,str(t.dtype),list(t.shape)]).encode())
        h.update(t.reshape(-1).view(torch.uint8).numpy().tobytes())
    return h.hexdigest()

In [2]:
torch.set_num_threads(1)
seed_all(8401)
RUN_FULL_EXPERIMENT = True
OUTPUT_ROOT = Path('lesson_84_outputs')
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__, 'CUDA可用:', torch.cuda.is_available())
print('本课使用CPU逻辑tick，不测GPU性能。')

Python: 3.13.5
PyTorch: 2.10.0+cpu CUDA可用: False
本课使用CPU逻辑tick，不测GPU性能。


## 1. 为什么是 Reconcile，而不是“每次看见差异都覆盖”？

Kubernetes 的控制循环周期性比较期望状态和观测状态，让实际逐渐趋近意图。这里的期望并不是管理员临时上传的字符串，而是**已经通过前一课发布事务的配置版本**。[Kubernetes 控制器概念](https://kubernetes.io/docs/concepts/architecture/controller/)

控制循环只负责三件事：观察、决定、执行受限动作。**它不会在发现别人改了配置时，自动创建新的配置 generation，也不会绕过上节的人工审批与双副本发布门槛。**

```text
发布事务的活动指针 → 当前期望 generation / SHA
                ↓
        A/B 实际配置＋模型身份＋WAL控制指针
                ↓
          分类：健康 / 可自动修 / 危险或未知
                ↓
       Ready / Grace / Backoff / Quarantine / Manual
```

*Kubernetes Server-Side Apply* 还有真正的字段所有权与冲突语义。本课只实现一条硬编码白名单，**不拥有 apiserver 的字段管理能力**。[SSA 字段所有权](https://kubernetes.io/docs/reference/using-api/server-side-apply/)

## 2. 用真实的小模型和持久 WAL 做实验

为让调谐器的决策有可核对的对象，下面保留上节自包含的因果模型、分页 KV 张量、两版 LoRA、不可变模型身份以及本地同步 WAL。

这些基础设施单元比较长，可以先运行，练习时重点看它们的**输入输出接口**；不是要求你在这一课重新写注意力算法。

教学模型随机初始化；KV 的一致性测试只验证数值执行路径，**不报告自然语言能力或真实模型部署速度**。

In [3]:
# %% 02 教学分词器、因果模型与LoRA（自包含）
class ByteMergeChatTokenizer:
    """自建教学分词器：控制token独立于用户文本，普通文本按UTF-8字节合并。

    不假装实现HF的FastTokenizer或任意模型Chat Template。固定控制ID，避免把用户
    输入里的字面控制符误当结构token。真实应用必须使用对应模型的官方tokenizer/template。
    """
    pad_token_id = 0
    bos_token_id = 1
    eos_token_id = 2
    im_start_token_id = 3
    im_end_token_id = 4
    role_ids = {"system":5, "user":6, "assistant":7}
    byte_offset = 16
    merges = (b"OK", b" 42", b"42", b"hello", b"world", b"\n", b"answer", b"JSON", b"{\"answer\":")

    def __init__(self):
        self.merge_ids = {piece: self.byte_offset + 256 + i for i,piece in enumerate(self.merges)}
        self.ids_to_merge = {v:k for k,v in self.merge_ids.items()}
        self.vocab_size = self.byte_offset + 256 + len(self.merges)
        self.specials = {0:"<pad>",1:"<bos>",2:"<eos>",3:"<im_start>",
                         4:"<im_end>",5:"<system>",6:"<user>",7:"<assistant>"}

    def encode(self, text: str, *, add_special_tokens: bool=False) -> list[int]:
        bounded_str(text, 4096, "文本")
        raw = text.encode("utf-8")
        result = []
        index = 0
        while index < len(raw):
            matches = [piece for piece in self.merges if raw.startswith(piece, index)]
            if matches:
                piece = max(matches, key=len)
                result.append(self.merge_ids[piece]);index += len(piece)
            else:
                result.append(self.byte_offset + raw[index]);index += 1
        return ([self.bos_token_id] if add_special_tokens else []) + result

    def decode(self, ids: list[int], *, skip_special_tokens: bool=False) -> str:
        require(type(ids) is list, "token IDs必须是列表")
        raw = bytearray()
        for i in ids:
            require(type(i) is int and 0 <= i < self.vocab_size, "token ID越界")
            if i in self.specials:
                if skip_special_tokens:
                    continue
                raise LessonError("正文decode不允许控制token；传入skip_special_tokens=True")
            if i in self.ids_to_merge: raw.extend(self.ids_to_merge[i])
            elif self.byte_offset <= i < self.byte_offset + 256: raw.append(i - self.byte_offset)
            else: raise LessonError("保留ID不是合法正文token")
        try:
            return bytes(raw).decode("utf-8",errors="strict")
        except UnicodeDecodeError as exc:
            raise LessonError("不完整UTF-8回答；不要用replacement字符伪装成功") from exc

    def apply_chat_template(self, messages: list[dict], *, tokenize: bool=True,
                            add_generation_prompt: bool=False):
        require(type(messages) is list and bool(messages), "messages需要非空列表")
        require(type(add_generation_prompt) is bool, "generation prompt必须是bool")
        ids = [self.bos_token_id]
        for j,m in enumerate(messages):
            require(type(m) is dict and set(m)=={"role","content"}, "message字段必须恰好role/content")
            require(m["role"] in self.role_ids and type(m["content"]) is str, "角色或内容非法")
            require(not (j == len(messages)-1 and add_generation_prompt and m["role"]=="assistant"),
                    "若最后已有assistant，请使用续写语义，不要再添加generation prompt")
            ids.extend([self.im_start_token_id,self.role_ids[m["role"]]])
            ids.extend(self.encode(m["content"],add_special_tokens=False))
            ids.append(self.eos_token_id if m["role"]=="assistant" else self.im_end_token_id)
        if add_generation_prompt:
            ids.extend([self.im_start_token_id,self.role_ids["assistant"]])
        if tokenize:
            return ids
        # 仅用于人类阅读，控制符是标记，不要将其当普通文本用encode()重新编码。
        visible = []
        for i in ids:
            if i in self.specials: visible.append(self.specials[i])
            else: visible.append(self.decode([i],skip_special_tokens=True) if
                                 self._token_is_full_utf8(i) else "<byte-part>")
        return "".join(visible)

    def _token_is_full_utf8(self, i):
        if i in self.ids_to_merge:
            try: self.ids_to_merge[i].decode("utf-8");return True
            except UnicodeError: return False
        return self.byte_offset <= i < self.byte_offset + 128

    def contract(self) -> dict:
        return {"type":"toy-byte-merge", "specials":self.specials,
                "merges":[p.hex() for p in self.merges],"vocab":self.vocab_size,
                "prefix":"BOS+message start/role/body+end+assistant header",
                "eos":self.eos_token_id,"pad":self.pad_token_id}


@dataclass(frozen=True)
class ToyConfig:
    vocab_size: int
    dim: int=32
    heads: int=4
    layers: int=2
    ff_dim: int=64
    max_length: int=160

    def validate(self):
        require(all(type(x) is int and x>0 for x in asdict(self).values()),'配置必须为正整数')
        require(self.dim%self.heads==0 and self.vocab_size>16,
                'dim/head或词表不匹配')


class CausalSelfAttention(nn.Module):
    def __init__(self,cfg: ToyConfig):
        super().__init__()
        d=cfg.dim
        self.heads=cfg.heads
        self.q_proj=nn.Linear(d,d,bias=False)
        self.k_proj=nn.Linear(d,d,bias=False)
        self.v_proj=nn.Linear(d,d,bias=False)
        self.o_proj=nn.Linear(d,d,bias=False)

    def forward(self,x:torch.Tensor,attention_mask:torch.Tensor):
        b,t,d=x.shape;h=self.heads
        def split(layer):
            return layer(x).reshape(b,t,h,d//h).transpose(1,2)
        q,k,v=split(self.q_proj),split(self.k_proj),split(self.v_proj)
        scores=(q@k.transpose(-1,-2))/math.sqrt(d//h)
        lower=torch.ones(t,t,device=x.device,dtype=torch.bool).tril()
        allowed=lower[None,None,:,:]&attention_mask[:,None,None,:]
        weights=torch.softmax(scores.masked_fill(~allowed,float('-inf')),dim=-1)
        result=(weights@v).transpose(1,2).contiguous().reshape(b,t,d)
        return self.o_proj(result)


class TransformerBlock(nn.Module):
    def __init__(self,cfg:ToyConfig):
        super().__init__()
        self.norm1=nn.LayerNorm(cfg.dim)
        self.attn=CausalSelfAttention(cfg)
        self.norm2=nn.LayerNorm(cfg.dim)
        self.ff=nn.Sequential(nn.Linear(cfg.dim,cfg.ff_dim),nn.GELU(),
                              nn.Linear(cfg.ff_dim,cfg.dim))

    def forward(self,x,mask):
        x=x+self.attn(self.norm1(x),mask)
        return x+self.ff(self.norm2(x))


class TinyChatLM(nn.Module):
    def __init__(self,cfg:ToyConfig):
        super().__init__();cfg.validate();self.cfg=cfg
        self.embed=nn.Embedding(cfg.vocab_size,cfg.dim)
        self.position=nn.Embedding(cfg.max_length,cfg.dim)
        self.blocks=nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.layers)])
        self.norm=nn.LayerNorm(cfg.dim)
        self.head=nn.Linear(cfg.dim,cfg.vocab_size,bias=False)

    def forward(self,input_ids:torch.Tensor,attention_mask:torch.Tensor):
        require(input_ids.ndim==2 and input_ids.dtype==torch.long,'输入必须为[B,T] long')
        b,t=input_ids.shape
        require(b>0 and t>=1 and t<=self.cfg.max_length,'形状或上下文超限')
        require(attention_mask.shape==input_ids.shape and attention_mask.dtype==torch.bool,
                'attention mask形状/type不符')
        require(bool(attention_mask[:,0].all()),'不接受左补齐/空提示')
        require(not bool(((~attention_mask[:,:-1])&attention_mask[:,1:]).any()),
                '只允许右补齐')
        positions=torch.arange(t,device=input_ids.device)
        x=self.embed(input_ids)+self.position(positions)[None]
        for layer in self.blocks:x=layer(x,attention_mask)
        return self.head(self.norm(x))


class LoRALinear(nn.Module):
    def __init__(self,base:nn.Linear,rank=4,alpha=8.0):
        super().__init__()
        require(type(base) is nn.Linear and type(rank) is int and rank>0,
                '手写LoRA仅支持nn.Linear与正整数rank')
        require(type(alpha) in (int,float) and math.isfinite(alpha) and alpha>0,
                'LoRA alpha非法')
        self.base=base
        self.base.requires_grad_(False)
        self.rank=rank;self.alpha=float(alpha);self.enabled=True
        self.A=nn.Parameter(base.weight.new_empty(rank,base.in_features))
        self.B=nn.Parameter(base.weight.new_zeros(base.out_features,rank))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))

    def forward(self,x):
        result=self.base(x)
        if not self.enabled:return result
        return result+(self.alpha/self.rank)*F.linear(F.linear(x,self.A),self.B)


def inject_lora(base:TinyChatLM,rank=4,alpha=8.0) -> TinyChatLM:
    require(not any(isinstance(m,LoRALinear) for m in base.modules()),
            '不允许重复注入LoRA')
    policy=copy.deepcopy(base)
    policy.requires_grad_(False)
    for block in policy.blocks:
        block.attn.q_proj=LoRALinear(block.attn.q_proj,rank,alpha)
        block.attn.v_proj=LoRALinear(block.attn.v_proj,rank,alpha)
    return policy


def adapters(model:TinyChatLM):
    return [(n,p) for n,p in model.named_parameters() if n.endswith(('.A','.B'))]


def adapter_snapshot(model:TinyChatLM):
    return {n:p.detach().cpu().clone() for n,p in adapters(model)}


def restore_adapter(model:TinyChatLM,values:dict):
    parameters=dict(model.named_parameters())
    require(set(values)==set(adapter_snapshot(model)),'adapter键集合不匹配')
    with torch.no_grad():
        for name,v in values.items():
            require(v.shape==parameters[name].shape and bool(torch.isfinite(v).all()),
                    'adapter参数形状/数值不匹配')
            parameters[name].copy_(v.to(parameters[name]))


def base_state(model:TinyChatLM):
    base={}
    for name,value in model.state_dict().items():
        if name.endswith(('.A','.B')):continue
        normalized=name.replace('.q_proj.base.','.q_proj.').replace('.v_proj.base.','.v_proj.')
        require(normalized not in base,'归一化基座命名冲突')
        base[normalized]=value
    return base


def base_hash(model:TinyChatLM):
    return state_hash(base_state(model))

In [4]:
# %% 03 实际KV物理缓存与增量重算（不直接持久化tensor）
@dataclass
class KVSequence:
    version: str
    tokens: list[int] = field(default_factory=list)
    block_table: list[int] = field(default_factory=list)
    pending: dict | None = None


class NoKVSpace(LessonError):
    """资源耗尽是明确失败，不能当成EOS、正常输出或偷偷丢弃KV。"""


class PagedKVPool:
    """实际CPU Tensor承载所有layer的K/V；块表是教学页索引而非vLLM kernel。

    backing: [block, layer, key_or_value, slot, kv_head, head_dim]。
    模型和版本不可在已有序列上切换；共享块只在相同版本下合法。
    尾块写入时若被共享，先Copy-on-Write复制整个块所有层再写新token。
    """
    def __init__(self, capacity_blocks: int, block_size: int, model_cfg: ToyConfig,
                 versions: set[str], dtype: torch.dtype=torch.float32):
        require(type(capacity_blocks) is int and capacity_blocks>0,'块容量必须是正整数')
        require(type(block_size) is int and block_size>0,'块token大小必须是正整数')
        model_cfg.validate()
        require(type(versions) is set and bool(versions) and all(type(v) is str and v for v in versions),
                '版本集合非法')
        require(dtype in {torch.float32,torch.float64},'教学池仅实现float32/float64')
        self.cfg=model_cfg;self.capacity_blocks=capacity_blocks;self.block_size=block_size
        self.versions=set(versions);self.dtype=dtype
        self.head_dim=model_cfg.dim//model_cfg.heads
        self.data=torch.zeros((capacity_blocks,model_cfg.layers,2,block_size,
                               model_cfg.heads,self.head_dim),dtype=dtype,device='cpu')
        self.refs=[0]*capacity_blocks
        self.free=list(range(capacity_blocks))
        self.sessions:dict[str,KVSequence]={}
        self.allocation_count=0;self.reuse_count=0;self.cow_count=0
        self.oom_count=0;self.peak_live_blocks=0;self.peak_sessions=0
        self.zeroized_releases=0

    @property
    def bytes_per_token(self):
        return 2*self.cfg.layers*self.cfg.heads*self.head_dim*self.data.element_size()

    @property
    def bytes_per_block(self):
        return self.bytes_per_token*self.block_size

    def _sid(self,sid):
        bounded_str(sid,48,'会话ID')
        require(bool(re.fullmatch(r'[A-Za-z0-9_-]+',sid)),'会话ID仅限字母数字_-')
        return sid

    def _alloc(self):
        if not self.free:
            self.oom_count+=1
            raise NoKVSpace('KV页池容量不足；拒绝本次操作且不静默覆盖缓存')
        bid=self.free.pop(0)
        require(self.refs[bid]==0,'空闲块引用数不为0')
        require(not bool(self.data[bid].count_nonzero()),'空闲块残留数据，拒绝泄漏')
        self.refs[bid]=1;self.allocation_count+=1
        self.peak_live_blocks=max(self.peak_live_blocks,self.capacity_blocks-len(self.free))
        return bid

    def _decref(self,bid):
        require(type(bid) is int and 0<=bid<self.capacity_blocks and self.refs[bid]>0,
                '引用计数下溢或块ID非法')
        self.refs[bid]-=1
        if self.refs[bid]==0:
            with torch.no_grad():self.data[bid].zero_()
            self.free.append(bid);self.free.sort();self.zeroized_releases+=1

    def open(self,sid:str,version:str):
        self._sid(sid)
        require(version in self.versions,'未知模型版本不能建缓存')
        require(sid not in self.sessions,'不能重复建同一会话')
        self.sessions[sid]=KVSequence(version=version)
        self.peak_sessions=max(self.peak_sessions,len(self.sessions))

    def fork(self,parent:str,child:str,version:str):
        self._sid(child)
        require(parent in self.sessions and child not in self.sessions,'fork父会话缺失或子会话已存在')
        p=self.sessions[parent]
        require(p.pending is None,'不能复制正在写入的缓存')
        require(version==p.version,'跨模型版本不能共享KV：即使基座相同adapter也可能不同')
        for bid in p.block_table:self.refs[bid]+=1
        self.sessions[child]=KVSequence(version=version,tokens=p.tokens.copy(),
                                        block_table=p.block_table.copy())
        self.peak_sessions=max(self.peak_sessions,len(self.sessions))
        self.audit()

    def begin(self,sid:str,token_id:int,max_length:int):
        require(sid in self.sessions,'不存在的会话')
        s=self.sessions[sid]
        require(s.pending is None,'同一会话不允许并发写入')
        require(type(token_id) is int and token_id>=0,'新token ID非法')
        require(type(max_length) is int and len(s.tokens)<max_length,'超过上下文长度')
        slot=len(s.tokens)%self.block_size
        new_page=False;cow_old=None
        # 先检查容量，错误时保留全部原有块表和引用数。
        if slot==0 or (slot!=0 and self.refs[s.block_table[-1]]>1):
            if not self.free:
                self.oom_count+=1
                raise NoKVSpace('新页或写时复制需要空闲块，但KV页池已满')
            new_page=True
        if new_page:
            bid=self._alloc()
            if slot==0:
                s.block_table.append(bid)
            else:
                cow_old=s.block_table[-1]
                with torch.no_grad():self.data[bid].copy_(self.data[cow_old])
                self._decref(cow_old)
                s.block_table[-1]=bid
                self.cow_count+=1
        else:
            bid=s.block_table[-1]
        s.pending={'token_id':token_id,'slot':slot,'block':bid,
                   'new_page':new_page,'cow_old':cow_old,'written':set()}

    def put_layer(self,sid:str,layer:int,k:torch.Tensor,v:torch.Tensor):
        require(sid in self.sessions and self.sessions[sid].pending is not None,'没有开始token写入')
        p=self.sessions[sid].pending
        require(type(layer) is int and 0<=layer<self.cfg.layers,'层ID非法')
        require(layer not in p['written'],'同一层重复写入')
        expected=(self.cfg.heads,self.head_dim)
        require(k.shape==expected and v.shape==expected and k.dtype==self.dtype and v.dtype==self.dtype,
                'K/V形状或dtype非法')
        require(k.device.type=='cpu' and v.device.type=='cpu','此教学页池只接受CPU K/V')
        require(bool(torch.isfinite(k).all()) and bool(torch.isfinite(v).all()),'K/V有非有限值')
        with torch.no_grad():
            self.data[p['block'],layer,0,p['slot']].copy_(k.detach())
            self.data[p['block'],layer,1,p['slot']].copy_(v.detach())
        p['written'].add(layer)

    def read_layer(self,sid:str,layer:int) -> tuple[torch.Tensor,torch.Tensor]:
        require(sid in self.sessions and self.sessions[sid].pending is not None,'read必须在token事务中')
        s=self.sessions[sid];p=s.pending
        require(layer in p['written'],'只能读取已写入当前token的层')
        length=len(s.tokens)+1
        ks=[];vs=[]
        for index,bid in enumerate(s.block_table):
            n=min(self.block_size,length-index*self.block_size)
            if n<=0:break
            # backing的slot轴在前，需要转成[heads,time,head_dim]。
            ks.append(self.data[bid,layer,0,:n].transpose(0,1))
            vs.append(self.data[bid,layer,1,:n].transpose(0,1))
        require(bool(ks) and sum(x.shape[1] for x in ks)==length,'KV块表与长度不同步')
        return torch.cat(ks,dim=1),torch.cat(vs,dim=1)

    def commit(self,sid):
        require(sid in self.sessions and self.sessions[sid].pending is not None,'无可提交的token')
        s=self.sessions[sid];p=s.pending
        require(p['written']==set(range(self.cfg.layers)),'有层尚未写入：禁止提交不完整KV')
        s.tokens.append(p['token_id']);s.pending=None
        self.audit()

    def abort(self,sid):
        require(sid in self.sessions and self.sessions[sid].pending is not None,'无可回滚的token')
        s=self.sessions[sid];p=s.pending
        bid=p['block'];slot=p['slot']
        if p['new_page']:
            if p['cow_old'] is not None:
                old=p['cow_old']
                s.block_table[-1]=old
                self.refs[old]+=1
            else:
                require(s.block_table[-1]==bid,'回滚末尾块不一致')
                s.block_table.pop()
            self._decref(bid)
        else:
            # 当前slot仍不属于任何已提交token，清空可能部分写入的K/V。
            with torch.no_grad():self.data[bid,:,:,slot].zero_()
        s.pending=None
        self.audit()

    def release(self,sid):
        require(sid in self.sessions,'重复释放或会话不存在')
        s=self.sessions[sid]
        require(s.pending is None,'请先回滚未完成写入再释放')
        for bid in s.block_table:self._decref(bid)
        del self.sessions[sid]
        self.audit()

    def usage(self) -> dict:
        table_ids=[b for s in self.sessions.values() for b in s.block_table]
        live=self.capacity_blocks-len(self.free)
        # 若block被两个会话引用，其有效token只计物理占用一次。
        filled={}
        for s in self.sessions.values():
            length=len(s.tokens)+(1 if s.pending else 0)
            for i,bid in enumerate(s.block_table):
                filled[bid]=max(filled.get(bid,0),min(self.block_size,length-i*self.block_size))
        occupied=sum(filled.values())
        logical_refs=len(table_ids)
        return {'capacity_blocks':self.capacity_blocks,'block_size':self.block_size,
                'free_blocks':len(self.free),'live_blocks':live,
                'physical_storage_bytes':self.data.numel()*self.data.element_size(),
                'bytes_per_token':self.bytes_per_token,'bytes_per_block':self.bytes_per_block,
                'live_reserved_bytes':live*self.bytes_per_block,
                'physical_used_slots':occupied,'live_content_bytes':occupied*self.bytes_per_token,
                'physical_internal_fragmentation_bytes':(live*self.block_size-occupied)*self.bytes_per_token,
                'logical_token_uses':sum(len(s.tokens) for s in self.sessions.values()),
                'logical_page_references':logical_refs,
                'saved_physical_blocks_by_sharing':logical_refs-live,
                'session_count':len(self.sessions),'peak_live_blocks':self.peak_live_blocks,
                'peak_sessions':self.peak_sessions,'allocation_count':self.allocation_count,
                'cow_count':self.cow_count,'oom_count':self.oom_count,
                'zeroized_releases':self.zeroized_releases}

    def audit(self) -> dict:
        expected=[0]*self.capacity_blocks
        for sid,s in self.sessions.items():
            require(s.version in self.versions,'会话绑定未知版本')
            require(len(s.block_table)==math.ceil((len(s.tokens)+(1 if s.pending else 0))/self.block_size),
                    '块表页数与有效上下文长度不一致')
            require(len(set(s.block_table))==len(s.block_table),'同一会话不能重复引用物理块')
            for bid in s.block_table:
                require(type(bid) is int and 0<=bid<self.capacity_blocks,'块表有非法物理页')
                expected[bid]+=1
        require(expected==self.refs,'物理块引用计数不等于块表的真实引用')
        require(set(self.free)=={i for i,r in enumerate(self.refs) if r==0},'free list/引用计数不一致')
        require(len(self.free)==len(set(self.free)),'空闲页重复入队')
        require(all(not bool(self.data[i].count_nonzero()) for i in self.free),
                '空闲块包含未清除的KV数据')
        u=self.usage()
        require(u['physical_internal_fragmentation_bytes']>=0,'内部碎片负数')
        require(u['live_content_bytes']<=u['live_reserved_bytes']<=u['physical_storage_bytes'],
                '实际/保留字节账目不守恒')
        return u


class CachedModelRunner:
    """一次处理一个新增token。prefill本课逐token执行，非生产高吞吐chunked prefill。

    同一层先写该token的K/V，再用块表读取全部历史K/V，对单query计算attention。
    该前缀计算和原模型full forward的掩码、位置、LoRA权重完全一致。
    """
    def __init__(self,tok:ByteMergeChatTokenizer,models:dict[str,TinyChatLM],pool:PagedKVPool):
        require(set(models)==pool.versions,'模型版本与池的版本集合不一致')
        require(all(m.cfg==pool.cfg for m in models.values()),'模型配置和池配置不匹配')
        require(pool.dtype==torch.float32,'本课模型采用float32，KV存储dtype必须一致')
        require(all(next(m.parameters()).device.type=='cpu' for m in models.values()),'教学KV池仅支持CPU')
        require(all(not p.requires_grad for m in models.values() for p in m.parameters()),
                'KV缓存仅用于冻结推理模型，不能偷偷反传训练')
        self.tok=tok;self.models=models;self.pool=pool
        self.last_logits:dict[str,torch.Tensor]={}
        self.tokens_processed=0;self.attention_query_key_products=0
        self.full_recompute_query_key_products=0
        self.model_hashes={name:state_hash(m.state_dict()) for name,m in models.items()}

    @torch.inference_mode()
    def step(self,sid:str,token_id:int) -> torch.Tensor:
        require(sid in self.pool.sessions,'cache step会话不存在')
        s=self.pool.sessions[sid]
        require(type(token_id) is int and 0<=token_id<self.pool.cfg.vocab_size,'token ID非法')
        length=len(s.tokens)
        require(length<self.pool.cfg.max_length,'上下文长度已满')
        self.pool.begin(sid,token_id,self.pool.cfg.max_length)
        model=self.models[s.version]
        try:
            ids=torch.tensor([[token_id]],dtype=torch.long)
            pos=torch.tensor([length],dtype=torch.long)
            x=model.embed(ids)+model.position(pos)[None]
            heads=self.pool.cfg.heads;head_dim=self.pool.head_dim
            for index,block in enumerate(model.blocks):
                z=block.norm1(x)
                q=block.attn.q_proj(z).reshape(heads,head_dim)
                k=block.attn.k_proj(z).reshape(heads,head_dim)
                v=block.attn.v_proj(z).reshape(heads,head_dim)
                self.pool.put_layer(sid,index,k,v)
                all_k,all_v=self.pool.read_layer(sid,index)
                # query:[heads,1,d]; keys:[heads,seq,d]；此query位于整个缓存最末尾，
                # 所以其可见范围恰好是全部已存的过去token与当前token。
                scores=(q[:,None,:]@all_k.transpose(-2,-1))/math.sqrt(head_dim)
                weights=torch.softmax(scores,dim=-1)
                value=(weights@all_v).reshape(1,1,self.pool.cfg.dim)
                x=x+block.attn.o_proj(value)
                x=x+block.ff(block.norm2(x))
                self.attention_query_key_products+=heads*(length+1)
                self.full_recompute_query_key_products+=heads*(length+1)**2
            next_logits=model.head(model.norm(x))[0,0].detach().clone()
            require(bool(torch.isfinite(next_logits).all()),'模型输出非有限logits')
            self.pool.commit(sid)
            self.last_logits[sid]=next_logits
            self.tokens_processed+=1
            return next_logits
        except Exception:
            if self.pool.sessions[sid].pending is not None:self.pool.abort(sid)
            raise

    def prefill(self,sid:str,ids:list[int]) -> torch.Tensor:
        require(type(ids) is list and bool(ids),'prefill需要非空token列表')
        require(all(type(i) is int and 0<=i<self.pool.cfg.vocab_size for i in ids),
                'prefill出现非法token')
        require(sid in self.pool.sessions,'prefill会话不存在')
        require(len(self.pool.sessions[sid].tokens)+len(ids)<=self.pool.cfg.max_length,
                'prefill不能越界或静默截断')
        last=None
        for token in ids:last=self.step(sid,token)
        return last

    def open_prefill(self,sid,version,ids):
        self.pool.open(sid,version)
        try:return self.prefill(sid,ids)
        except Exception:
            self.close(sid)
            raise

    def fork(self,parent,child,version):
        require(parent in self.last_logits,'父序列尚未prefill，不能fork')
        self.pool.fork(parent,child,version)
        self.last_logits[child]=self.last_logits[parent].clone()

    def close(self,sid):
        self.pool.release(sid)
        self.last_logits.pop(sid,None)

    @torch.inference_mode()
    def full_last(self,version:str,ids:list[int]) -> torch.Tensor:
        require(version in self.models and bool(ids),'版本或输入非法')
        model=self.models[version]
        inp=torch.tensor([ids],dtype=torch.long)
        mask=torch.ones_like(inp,dtype=torch.bool)
        return model(inp,mask)[0,-1].detach().clone()

    def assert_consistent(self,sid,atol=2e-5):
        require(sid in self.pool.sessions and sid in self.last_logits,'尚无缓存输出')
        s=self.pool.sessions[sid]
        expected=self.full_last(s.version,s.tokens)
        got=self.last_logits[sid]
        max_delta=float((expected-got).abs().max())
        require(torch.allclose(expected,got,atol=atol,rtol=2e-5),
                '真实cache输出与full recompute不一致：检查位置、KV存储或因果掩码')
        return max_delta

In [5]:
# %% 04 双版本确定性模型与不可变身份
def build_servable_models(seed=7901):
    require(type(seed) is int and seed>=0,'seed非法')
    seed_all(seed)
    tok=ByteMergeChatTokenizer()
    cfg=ToyConfig(tok.vocab_size,dim=16,heads=2,layers=1,ff_dim=32,max_length=96)
    base=TinyChatLM(cfg).eval()
    stable=inject_lora(base,rank=4,alpha=8.0)
    canary=inject_lora(base,rank=4,alpha=8.0)
    gen=torch.Generator().manual_seed(seed+1)
    with torch.no_grad():
        for name,p in adapters(canary):
            if name.endswith('.B'):
                p.copy_(torch.randn(p.shape,generator=gen)*.1)
    models={'stable':stable.eval().requires_grad_(False),'canary':canary.eval().requires_grad_(False)}
    require(base_hash(models['stable'])==base_hash(models['canary']),'不是同一基座')
    require(state_hash(models['stable'].state_dict())!=state_hash(models['canary'].state_dict()),'两个版本实际权重相同')
    return tok,models

In [6]:
# %% 05 WAL逻辑事件、完整性检查与快照

ZERO_HASH='0'*64

@dataclass
class LogicalRequest:
    rid: str
    tenant: str
    key: str
    version: str
    prompt_ids: list[int]
    max_new_tokens: int
    generated: list[int]=field(default_factory=list)
    terminal: str|None=None

    def identity(self):
        return {'tenant':self.tenant,'key':self.key,'version':self.version,
                'prompt_ids':self.prompt_ids,'max_new_tokens':self.max_new_tokens}


class Ledger:
    """WAL顺序应用的逻辑状态，不包含任何模型权重或物理KV。"""
    def __init__(self,codec:ByteMergeChatTokenizer,model_identity:dict):
        self.codec=codec
        self.model_identity=copy.deepcopy(model_identity)
        self.sessions:dict[str,LogicalRequest]={}
        self.seq=0;self.last_hash=ZERO_HASH;self.tick=0

    def to_json(self):
        return {'seq':self.seq,'last_hash':self.last_hash,'tick':self.tick,
                'sessions':{key:asdict(state) for key,state in sorted(self.sessions.items())}}

    def apply(self,record:dict):
        require(type(record) is dict and set(record)=={'seq','tick','kind','payload','prev','hash'},
                'WAL字段集合错误')
        require(type(record['seq']) is int and record['seq']==self.seq+1,'WAL序号必须连续')
        require(type(record['tick']) is int and record['tick']==self.tick+1,
                '本课tick是从1开始的每条持久事件序号')
        require(record['prev']==self.last_hash,'WAL前驱hash错误或从错误分叉接入')
        require(record['hash']==digest({k:v for k,v in record.items() if k!='hash'}),
                'WAL内容hash错误')
        kind=record['kind'];p=record['payload']
        require(type(p) is dict,'payload必须是dict')
        if kind=='create':
            require(set(p)=={'rid','tenant','key','version','prompt_ids','max_new_tokens'},
                    'create字段集合错误')
            rid=make_rid(p['tenant'],p['key'])
            require(rid==p['rid'] and rid not in self.sessions,'重复idempotency key/create冲突')
            require(p['version'] in self.model_identity['weights'],'WAL引用未知权重版本')
            require(type(p['prompt_ids']) is list and len(p['prompt_ids'])>0 and
                    len(p['prompt_ids'])<self.model_identity['max_length'], '提示token非法/超上下文')
            require(all(type(t) is int and 0<=t<self.codec.vocab_size for t in p['prompt_ids']),
                    'WAL提示token超词表')
            require(type(p['max_new_tokens']) is int and 1<=p['max_new_tokens']<=4 and
                    len(p['prompt_ids'])+p['max_new_tokens']<=self.model_identity['max_length'],
                    'WAL完成长度非法')
            self.sessions[rid]=LogicalRequest(**p)
        elif kind=='token':
            require(set(p)=={'rid','index','token','terminal'},'token字段集合错误')
            rid=p['rid'];require(rid in self.sessions,'追加token的请求不存在')
            session=self.sessions[rid]
            require(session.terminal is None,'终态后不可再追加token')
            require(type(p['index']) is int and p['index']==len(session.generated),
                    'token序号不连续')
            tok=p['token'];require(type(tok) is int and 0<=tok<self.codec.vocab_size,
                                  'token ID越界')
            require(p['terminal'] in {None,'eos','length'},'非法停止原因')
            next_length=len(session.generated)+1
            expected='eos' if tok==self.codec.eos_token_id else (
                     'length' if next_length==session.max_new_tokens else None)
            require(p['terminal']==expected,'EOS、长度终止不一致')
            session.generated.append(tok)
            session.terminal=p['terminal']
        else:
            raise LessonError('WAL事件类型非法')
        self.seq=record['seq'];self.tick=record['tick'];self.last_hash=record['hash']

    @classmethod
    def from_json(cls,codec,model_identity,data):
        require(type(data) is dict and set(data)=={'seq','last_hash','tick','sessions'},
                '快照状态字段错误')
        out=cls(codec,model_identity)
        require(type(data['seq']) is int and data['seq']>=0 and
                data['tick']==data['seq'] and type(data['last_hash']) is str and
                len(data['last_hash'])==64,'快照head错误')
        out.seq=data['seq'];out.tick=data['tick'];out.last_hash=data['last_hash']
        require(type(data['sessions']) is dict,'快照会话表必须为dict')
        for rid,s in data['sessions'].items():
            require(type(s) is dict and set(s)==set(LogicalRequest.__dataclass_fields__),
                    '快照会话字段非法')
            item=LogicalRequest(**s)
            require(item.rid==rid==make_rid(item.tenant,item.key),'快照id不一致')
            require(item.version in model_identity['weights'],'快照版本已失效')
            require(item.terminal in {None,'eos','length'},'快照停止状态非法')
            require(all(type(i) is int and 0<=i<codec.vocab_size for i in
                        item.prompt_ids+item.generated),'快照token越界')
            require(1<=item.max_new_tokens<=4 and
                    len(item.generated)<=item.max_new_tokens,'快照生成长度非法')
            if item.terminal=='eos':
                require(bool(item.generated) and item.generated[-1]==codec.eos_token_id,
                        '快照EOS状态不一致')
            if item.terminal=='length':
                require(len(item.generated)==item.max_new_tokens and
                        item.generated[-1]!=codec.eos_token_id,'快照长度终止不一致')
            if item.terminal is None:
                require(codec.eos_token_id not in item.generated and
                        len(item.generated)<item.max_new_tokens,'活动快照包含终态回答')
            out.sessions[rid]=item
        return out


def make_rid(tenant:str,key:str) -> str:
    for value,label in ((tenant,'租户'),(key,'幂等键')):
        require(type(value) is str and re.fullmatch('[A-Za-z0-9_-]{1,18}',value) is not None,
                label+'只能含1..18个ASCII字母数字_-')
    return tenant+'__'+key


def atomic_json(path:Path,payload:dict) -> None:
    """单机持久文件替换＋fsync；没有提供跨机器线性一致性。"""
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp')
    require(not tmp.exists(),'存在未清理的临时文件，拒绝覆盖')
    try:
        with tmp.open('x',encoding='utf-8') as f:
            f.write(json.dumps(payload,ensure_ascii=False,sort_keys=True,allow_nan=False)+'\n')
            f.flush();os.fsync(f.fileno())
        os.replace(tmp,path)
        if hasattr(os,'O_DIRECTORY'):
            fd=os.open(str(path.parent),os.O_RDONLY|os.O_DIRECTORY)
            try:os.fsync(fd)
            finally:os.close(fd)
    finally:
        if tmp.exists():tmp.unlink()


def make_record(ledger:Ledger,kind:str,payload:dict) -> dict:
    row={'seq':ledger.seq+1,'tick':ledger.tick+1,'kind':kind,
         'payload':copy.deepcopy(payload),'prev':ledger.last_hash}
    row['hash']=digest(row)
    trial=copy.deepcopy(ledger)
    trial.apply(row)
    return row


class DurableJournal:
    """严格顺序WAL：每行必须换行、hash chain，追加时flush+fsync。

    这是本地模拟“两个独立实例的磁盘”，不是分布式事务，不跨网复制fsync保证。
    故障后的未完成行默认拒绝；不静默截断或忽略。
    """
    def __init__(self,folder:Path,codec,model_identity,load_existing=False):
        self.folder=Path(folder);self.folder.mkdir(parents=True,exist_ok=True)
        self.path=self.folder/'events.jsonl'
        self.snapshot_path=self.folder/'snapshot.json'
        self.codec=codec;self.model_identity=copy.deepcopy(model_identity)
        if not load_existing:
            require(not self.path.exists() and not self.snapshot_path.exists(),
                    '实例目录已有数据，禁止当作新副本覆盖')
            self.path.write_bytes(b'')
        else:
            require(self.path.is_file(),'恢复时WAL不存在')
        self.records=[];self.ledger=Ledger(codec,model_identity)
        if load_existing:self.recover()

    def read_records(self) -> list[dict]:
        raw=self.path.read_bytes()
        require(not raw or raw.endswith(b'\n'),'WAL末尾存在截断行，必须人工修复/显式恢复流程')
        rows=[];temp=Ledger(self.codec,self.model_identity)
        for n,line in enumerate(raw.splitlines(),1):
            require(line!=b'','WAL存在空行')
            try:
                row=json.loads(line.decode('utf-8'))
            except (UnicodeDecodeError,json.JSONDecodeError) as exc:
                raise LessonError(f'WAL第{n}行不是有效JSON/UTF8') from exc
            temp.apply(row)
            rows.append(row)
        return rows

    def recover(self) -> dict:
        rows=self.read_records()  # 对全WAL做语法/hash链验证，快照不绕过完整性审计。
        all_state=Ledger(self.codec,self.model_identity)
        for row in rows:all_state.apply(row)
        snapshot_used=False;tail=rows
        if self.snapshot_path.exists():
            try:wrapped=json.loads(self.snapshot_path.read_text(encoding='utf-8'))
            except (UnicodeDecodeError,json.JSONDecodeError) as exc:
                raise LessonError('快照文件无法解码') from exc
            require(type(wrapped) is dict and set(wrapped)=={'payload','sha256'},'快照封装非法')
            data=wrapped['payload']
            require(wrapped['sha256']==digest(data),'快照内容摘要不匹配')
            require(type(data) is dict and set(data)=={'version','model_identity','state'},
                    '快照字段错误')
            require(data['version']==VERSION and data['model_identity']==self.model_identity,
                    '快照模型/分词身份不匹配')
            restored=Ledger.from_json(self.codec,self.model_identity,data['state'])
            require(restored.seq<=len(rows),'快照seq超出WAL实际长度')
            prefix=Ledger(self.codec,self.model_identity)
            for row in rows[:restored.seq]:prefix.apply(row)
            require(prefix.to_json()==restored.to_json(),
                    '快照与WAL前缀的内容不一致：不允许信任重算hash后的假快照')
            tail=rows[restored.seq:]
            for row in tail:restored.apply(row)
            require(restored.to_json()==all_state.to_json(),'快照加尾WAL不等于全量恢复')
            self.ledger=restored;snapshot_used=True
        else:self.ledger=all_state
        self.records=rows
        return {'wal_records_verified':len(rows),'snapshot_used':snapshot_used,
                'tail_events_semantically_applied':len(tail),'head_hash':self.ledger.last_hash}

    def append_row(self,row:dict) -> None:
        # 在触盘前先检查下一状态，避免错误事件落盘后再被拒绝。
        probe=copy.deepcopy(self.ledger);probe.apply(row)
        with self.path.open('ab') as f:
            payload=(canonical(row)+'\n').encode('utf-8')
            f.write(payload);f.flush();os.fsync(f.fileno())
        self.ledger=probe
        self.records.append(copy.deepcopy(row))

    def checkpoint(self) -> dict:
        data={'version':VERSION,'model_identity':copy.deepcopy(self.model_identity),
              'state':self.ledger.to_json()}
        atomic_json(self.snapshot_path,{'payload':data,'sha256':digest(data)})
        return {'seq':self.ledger.seq,'last_hash':self.ledger.last_hash,
                'size_bytes':self.snapshot_path.stat().st_size,
                'contents':'logical token IDs and request states; NOT raw KV tensors'}

In [7]:
# %% 06 独立推理副本：WAL恢复后重新prefill真实KV

def artifact_identity(tok,models:dict[str,TinyChatLM]) -> dict:
    require(type(models) is dict and set(models)=={'stable','canary'},'模型版本集合不完整')
    configs=[asdict(m.cfg) for m in models.values()]
    require(configs[0]==configs[1],'两个版本配置不同')
    return {'tokenizer':digest(tok.contract()),'weights':{k:state_hash(m.state_dict())
              for k,m in sorted(models.items())},'config':configs[0],
              'max_length':configs[0]['max_length']}


class Replica:
    def __init__(self,name:str,folder:Path,tok,models,capacity_blocks=64,
                 block_size=4,load_existing=False):
        require(name in {'A','B'},'副本名必须A/B')
        self.name=name;self.tok=tok;self.models=copy.deepcopy(models)
        self.identity=artifact_identity(tok,self.models)
        self.folder=Path(folder)
        self.journal=DurableJournal(self.folder,tok,self.identity,load_existing)
        cfg=next(iter(self.models.values())).cfg
        self.pool=PagedKVPool(capacity_blocks,block_size,cfg,set(self.models))
        self.runner=CachedModelRunner(tok,self.models,self.pool)
        self.up=True
        self.last_recovery={'wal_records_verified':0,'snapshot_used':False,
                            'tail_events_semantically_applied':0}
        if load_existing:self.recover_cache()

    @property
    def ledger(self):return self.journal.ledger

    def _expected(self,rid):
        require(rid in self.pool.sessions and rid in self.runner.last_logits,
                '活动请求没有对应真实KV；拒绝偷偷从空上下文解码')
        logits=self.runner.last_logits[rid]
        require(bool(torch.isfinite(logits).all()),'KV输出异常')
        return int(logits.argmax())

    def prevalidate(self,row:dict) -> None:
        require(self.up,'实例已宕机')
        require(artifact_identity(self.tok,self.models)==self.identity,
                '运行模型权重漂移，拒绝生成/应用新事件')
        test=copy.deepcopy(self.ledger);test.apply(row)
        kind=row['kind'];payload=row['payload']
        if kind=='token':
            rid=payload['rid']
            require(rid in self.pool.sessions and rid in self.runner.last_logits,
                    '副本缺失活动KV：先恢复再继续')
            require(self._expected(rid)==payload['token'],
                    '复制到另一个副本时增量输出不一致：权重/前缀/位置存在漂移')

    def accept(self,row:dict) -> None:
        self.prevalidate(row)
        self.journal.append_row(row)
        p=row['payload']
        try:
            if row['kind']=='create':
                self.runner.open_prefill(p['rid'],p['version'],p['prompt_ids'])
            else:
                if p['terminal'] is None:self.runner.step(p['rid'],p['token'])
                else:self.runner.close(p['rid'])
            self.pool.audit()
        except Exception:
            # 已持久事件不能撒谎称“未发生”；该副本必须标记不可用并离线重建。
            self.up=False
            raise

    def create(self,payload:dict) -> dict:
        record=make_record(self.ledger,'create',payload)
        self.accept(record)
        return record

    def token(self,payload:dict) -> dict:
        record=make_record(self.ledger,'token',payload)
        self.accept(record)
        return record

    def recover_cache(self) -> dict:
        require(artifact_identity(self.tok,self.models)==self.identity,
                '恢复实例权重身份不匹配')
        details=self.journal.recover()
        # 丢弃所有易失KV，依据逻辑会话从最早提示重建。
        cfg=next(iter(self.models.values())).cfg
        self.pool=PagedKVPool(self.pool.capacity_blocks,self.pool.block_size,cfg,set(self.models))
        self.runner=CachedModelRunner(self.tok,self.models,self.pool)
        total=0;active=0
        for rid,state in sorted(self.ledger.sessions.items()):
            if state.terminal is not None:continue
            active+=1
            self.runner.open_prefill(rid,state.version,state.prompt_ids)
            total+=len(state.prompt_ids)
            for t in state.generated:
                require(self._expected(rid)==t,
                        '恢复后同一前缀的贪心输出不同，拒绝身份/模型漂移')
                self.runner.step(rid,t)
                total+=1
            self.runner.assert_consistent(rid)
        self.up=True
        details.update({'active_sessions_rebuilt':active,'tokens_recomputed':total,
                        'kv_tensors_restored_from_disk':False,'kv_live_blocks':self.pool.usage()['live_blocks']})
        self.last_recovery=details
        return details

    def audit(self):
        active={k for k,v in self.ledger.sessions.items() if v.terminal is None}
        require(set(self.pool.sessions)==active and set(self.runner.last_logits)==active,
                '逻辑活跃会话与KV实体不一致')
        max_diff=0.0
        for rid in sorted(active):
            st=self.ledger.sessions[rid]
            require(self.pool.sessions[rid].tokens==st.prompt_ids+st.generated,
                    'KV与已确认的逻辑token历史不一致')
            require(self.pool.sessions[rid].version==st.version,'KV与持久版本绑定不一致')
            max_diff=max(max_diff,self.runner.assert_consistent(rid))
        u=self.pool.audit()
        return {'replica':self.name,'logical_seq':self.ledger.seq,'logical_requests':len(self.ledger.sessions),
                'active_sessions':len(active),'kv_live_blocks':u['live_blocks'],
                'max_cache_full_logits_diff':max_diff,'models':self.identity['weights']}


def _request_payload(tok,tenant,key,version,text,max_new_tokens):
    rid=make_rid(tenant,key)
    require(version in {'stable','canary'},'模型版本非法')
    require(type(max_new_tokens) is int and 1<=max_new_tokens<=4,'输出预算非法')
    prompt=tok.apply_chat_template([{'role':'user','content':text}],tokenize=True,
                                   add_generation_prompt=True)
    return {'rid':rid,'tenant':tenant,'key':key,'version':version,
            'prompt_ids':prompt,'max_new_tokens':max_new_tokens}

In [8]:
# %% 07 单控制点epoch围栏、同步/异步确认与接管

class SimulatedCrash(RuntimeError):
    """明确模拟写盘之后、客户端收到ACK之前的中断，不执行真实进程杀死。"""


def save_model_artifacts(path:Path,tok,models) -> dict:
    require(not Path(path).exists(),'模型artifact不可覆盖')
    identity=artifact_identity(tok,models)
    blob={'version':VERSION,'identity':identity,'tokenizer':tok.contract(),
          'weights':{version:{k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
                     for version,model in models.items()}}
    torch.save(blob,path)
    return identity


def load_model_artifacts(path:Path):
    # 仅加载本课自己生成、可信来源的纯张量与dict；不执行外部网络下载。
    blob=torch.load(path,map_location='cpu',weights_only=True)
    require(blob['version']==VERSION,'artifact版本不匹配')
    tok=ByteMergeChatTokenizer()
    require(blob['tokenizer']==tok.contract(),'tokenizer/chat template身份不匹配')
    identity=blob['identity']
    cfg=ToyConfig(**identity['config'])
    models={}
    for name,values in blob['weights'].items():
        model=inject_lora(TinyChatLM(cfg),rank=4,alpha=8.0)
        model.load_state_dict(values,strict=True)
        models[name]=model.eval().requires_grad_(False)
    require(artifact_identity(tok,models)==identity,'模型artifact哈希不匹配')
    return tok,models


class Coordinator:
    """一个教学用串行控制器，持久化epoch与ACK审计；不是分布式共识。"""
    def __init__(self,root:Path,tok,models,policy='sync',seed=8101,
                 capacity_blocks=64,load_existing=False):
        require(policy in {'sync','async'},'确认模式只能sync/async')
        self.root=Path(root);self.root.mkdir(parents=True,exist_ok=True)
        self.control_path=self.root/'control.json'
        self.artifact_path=self.root/'models.pt'
        self.tok=tok;self.models=models
        self.model_identity=artifact_identity(tok,models)
        self.capacity_blocks=capacity_blocks
        self.seed=seed
        if load_existing:
            self.control=self._read_control()
            require(self.control['model_identity']==self.model_identity,
                    '重载模型与原先已确认事件的权重/词表身份不符')
            require(self.control['policy']==policy,'控制器确认策略不可暗中更改')
        else:
            require(not self.control_path.exists() and not self.artifact_path.exists(),
                    '根目录已包含集群持久状态，禁止覆盖')
            save_model_artifacts(self.artifact_path,tok,models)
            self.control={'version':VERSION,'active':'A','epoch':1,'policy':policy,
                          'model_identity':self.model_identity,'ack_seq':0,'ack_tick':0,
                          'ack_history':[],'failovers':[],'quarantined':[],'seed':seed,
                          'capacity_blocks':capacity_blocks}
            atomic_json(self.control_path,self.control)
        self.rep={name:Replica(name,self.root/name,tok,models,capacity_blocks,
                               load_existing=load_existing) for name in ('A','B')}
        self.updates_acknowledged=0
        for name in self.control['quarantined']:
            require(name!=self.control['active'],'活跃主不能被隔离')
            self.rep[name].shutdown()
        self._check_artifacts()

    def _read_control(self):
        try:state=json.loads(self.control_path.read_text(encoding='utf-8'))
        except (UnicodeDecodeError,json.JSONDecodeError,FileNotFoundError) as exc:
            raise LessonError('控制指针无法读取') from exc
        require(type(state) is dict and
                set(state)=={'version','active','epoch','policy','model_identity','ack_seq',
                             'ack_tick','ack_history','failovers','quarantined','seed','capacity_blocks'},
                '控制记录字段非法')
        require(state['version']==VERSION and state['active'] in {'A','B'} and
                type(state['epoch']) is int and state['epoch']>=1,'控制代际非法')
        require(type(state['ack_seq']) is int and state['ack_seq']>=0 and
                state['ack_tick']==state['ack_seq'],'ACK水位非法')
        require(type(state['quarantined']) is list and
                set(state['quarantined'])<={'A','B'} and
                state['active'] not in state['quarantined'],
                '隔离副本列表非法')
        return state

    def _check_artifacts(self):
        token,models=load_model_artifacts(self.artifact_path)
        require(artifact_identity(token,models)==self.model_identity,
                '模型artifact与当前控制身份不符')
        for r in self.rep.values():
            require(r.identity==self.model_identity,'副本模型身份漂移')
            require(artifact_identity(r.tok,r.models)==r.identity,'运行中模型权重被改写')

    def _fence(self,epoch:int):
        durable=self._read_control()
        require(type(epoch) is int and epoch==durable['epoch'] and
                durable['active']==self.control['active'] and
                durable['epoch']==self.control['epoch'],
                'epoch过期或控制指针已被其他控制器切换；拒绝陈旧主节点写入')
        require(self.rep[durable['active']].up,'活跃副本不可用')
        return self.rep[durable['active']]

    def _standby(self):return self.rep['B' if self.control['active']=='A' else 'A']

    def replicate_pending(self):
        primary=self.rep[self.control['active']];target=self._standby()
        require(target.name not in self.control['quarantined'],
                '备机日志已隔离，必须显式丢弃旧分叉并重建')
        require(target.up,'备机不可用，不能确认同步复制')
        require(target.identity==primary.identity,'复制副本模型身份不相等')
        left=target.journal.records;right=primary.journal.records
        require(len(left)<=len(right) and
                all(a==b for a,b in zip(left,right)),
                '日志发生分叉：不可自动拼接，必须隔离并人工处理')
        count=0
        for row in right[len(left):]:
            target.accept(row);count+=1
        require(target.ledger.last_hash==primary.journal.records[target.ledger.seq-1]['hash']
                if target.ledger.seq else target.ledger.last_hash==ZERO_HASH,
                '复制完成后head hash不一致')
        return count

    def _ack(self,row:dict):
        require(type(row) is dict and 'seq' in row and 'hash' in row,'ACK没有WAL证据')
        primary=self.rep[self.control['active']]
        require(primary.ledger.seq>=row['seq'] and
                primary.journal.records[row['seq']-1]==row,'主机日志和ACK不匹配')
        already_acked=row['seq']<=self.control['ack_seq']
        if already_acked:
            require(any(e['seq']==row['seq'] and e['hash']==row['hash']
                        for e in self.control['ack_history']),
                    '历史ACK证据与当前日志不匹配')
        elif self.control['policy']=='sync':
            standby=self._standby()
            require(standby.name not in self.control['quarantined'] and
                    standby.up and standby.ledger.seq>=row['seq'] and
                    standby.journal.records[row['seq']-1]==row,'同步ACK前备机没有持久化该事件')
        if row['seq']>self.control['ack_seq']:
            require(row['seq']==self.control['ack_seq']+1,'不能越过未确认的先前事件')
            self.control['ack_seq']=row['seq'];self.control['ack_tick']=row['tick']
            self.control['ack_history'].append({'seq':row['seq'],'hash':row['hash'],
                                               'epoch':self.control['epoch']})
            atomic_json(self.control_path,self.control)
            self.updates_acknowledged+=1
        return {'seq':row['seq'],'hash':row['hash'],'epoch':self.control['epoch'],
                'durability':'primary+standby' if self.control['policy']=='sync' else 'primary_only'}

    def _commit(self,row:dict,crash_after_durable=False):
        primary=self.rep[self.control['active']]
        primary.accept(row)   # 先写盘fsync，再响应或复制。
        if self.control['policy']=='sync':self.replicate_pending()
        if crash_after_durable:
            raise SimulatedCrash('WAL已fsync但控制ACK未写入；客户端必须同键对账重试')
        return self._ack(row)

    def start(self,epoch,tenant,key,version='stable',text='hello',max_new_tokens=4,
              crash_after_durable=False):
        active=self._fence(epoch)
        payload=_request_payload(self.tok,tenant,key,version,text,max_new_tokens)
        rid=payload['rid']
        if rid in active.ledger.sessions:
            item=active.ledger.sessions[rid]
            require(item.identity()=={k:payload[k] for k in item.identity()},
                    '同一幂等键与先前请求内容/版本不一致')
            row=next(r for r in active.journal.records if r['kind']=='create'
                     and r['payload']['rid']==rid)
            if self.control['policy']=='sync' and row['seq']>self.control['ack_seq']:
                self.replicate_pending()
            ack=self._ack(row)
            return {'rid':rid,'new_event':False,'ack':ack}
        row=make_record(active.ledger,'create',payload)
        ack=self._commit(row,crash_after_durable)
        return {'rid':rid,'new_event':True,'ack':ack}

    def step(self,epoch,rid,expected_index:int,crash_after_durable=False):
        active=self._fence(epoch)
        require(rid in active.ledger.sessions,'请求不存在')
        item=active.ledger.sessions[rid]
        require(type(expected_index) is int and expected_index>=0,'续传序号非法')
        if expected_index<len(item.generated):
            row=next(r for r in active.journal.records if r['kind']=='token' and
                     r['payload']['rid']==rid and r['payload']['index']==expected_index)
            if self.control['policy']=='sync' and row['seq']>self.control['ack_seq']:
                self.replicate_pending()
            ack=self._ack(row)
            return {'rid':rid,'index':expected_index,'token':row['payload']['token'],
                    'terminal':row['payload']['terminal'],'new_event':False,'ack':ack}
        require(expected_index==len(item.generated),'不能跳过尚未产生的token')
        require(item.terminal is None,'请求已经终止')
        tok=active._expected(rid)
        stop='eos' if tok==self.tok.eos_token_id else (
             'length' if expected_index+1==item.max_new_tokens else None)
        row=make_record(active.ledger,'token',{'rid':rid,'index':expected_index,
                                              'token':tok,'terminal':stop})
        ack=self._commit(row,crash_after_durable)
        return {'rid':rid,'index':expected_index,'token':tok,'terminal':stop,
                'new_event':True,'ack':ack}

    def checkpoint(self):
        return {name:self.rep[name].journal.checkpoint() for name in ('A','B')}

    def promote(self,epoch,target='B',allow_data_loss=False,rpo_budget_ticks=0,
                rto_budget_ticks=12,tokens_per_tick=32,wal_records_per_tick=16):
        require(type(allow_data_loss) is bool,'数据丢失授权必须显式bool')
        require(target in self.rep and target!=self.control['active'],'接管目标非法')
        persisted=self._read_control()
        require(type(epoch) is int and epoch==self.control['epoch'] and
                persisted==self.control, '接管用epoch过期或控制状态发生变化')
        # 故障后原主机可down；只有这个教学控制指针继续提供围栏，不由原主授权。
        require(all(type(v) is int and v>=0 for v in (rpo_budget_ticks,rto_budget_ticks))
                and tokens_per_tick>0 and wal_records_per_tick>0,'恢复目标/预算非法')
        old_active=self.control['active'];standby=self.rep[target]
        require(standby.up,'备机不可用')
        self._check_artifacts()
        # 在新对象中只从本地持久日志及快照恢复，易失KV不复制过来。
        fresh=Replica(target,self.root/target,self.tok,self.models,self.capacity_blocks,
                      load_existing=True)
        require(artifact_identity(fresh.tok,fresh.models)==self.model_identity,
                '灾难恢复模型/模板指纹不一致')
        audit=fresh.audit()
        ack_seq=self.control['ack_seq']
        lost=max(0,ack_seq-fresh.ledger.seq)
        require(fresh.ledger.seq<=self.rep[old_active].ledger.seq,
                '备机不能凭空比主机多出旧epoch事件')
        # 完整一致的确认前缀不能只是seq相等；还要比较哈希链前缀。
        primary_records=self.rep[old_active].journal.records
        require(fresh.journal.records==primary_records[:len(fresh.journal.records)],
                '备机历史并非旧主已提交前缀；禁止接管分叉日志')
        rpo_gap=max(0,self.control['ack_tick']-fresh.ledger.tick)
        rto=2+math.ceil(fresh.last_recovery['wal_records_verified']/wal_records_per_tick)+(
             math.ceil(fresh.last_recovery['tokens_recomputed']/tokens_per_tick))
        if (lost>0 or rpo_gap>rpo_budget_ticks) and not allow_data_loss:
            raise LessonError('未复制的已ACK事件存在，自动接管必须阻断并报告RPO风险')
        if rto>rto_budget_ticks:
            raise LessonError('预计恢复逻辑tick超出RTO门槛；本课拒绝自动接管')
        report={'from':old_active,'to':target,'old_epoch':epoch,'new_epoch':epoch+1,
                'sync_mode':self.control['policy'],'ack_seq_before':ack_seq,
                'target_seq':fresh.ledger.seq,'lost_acked_events':lost,
                'rpo_gap_logical_ticks':rpo_gap,'rpo_objective_ticks':rpo_budget_ticks,
                'rto_simulated_ticks':rto,'rto_objective_ticks':rto_budget_ticks,
                'rto_formula':'2 + ceil(WAL_verified / records_per_tick) + ceil(KV_replay_tokens / tokens_per_tick)',
                'recovered_active_sessions':audit['active_sessions'],
                'recovered_tokens_recomputed':fresh.last_recovery['tokens_recomputed'],
                'data_loss_override':allow_data_loss and lost>0,
                'no_real_time_measurement':True}
        new_control=copy.deepcopy(self.control)
        new_control['active']=target;new_control['epoch']=epoch+1
        new_control['ack_seq']=fresh.ledger.seq;new_control['ack_tick']=fresh.ledger.tick
        if lost>0:
            new_control['ack_history']=[row for row in new_control['ack_history']
                                        if row['seq']<=fresh.ledger.seq]
            new_control['quarantined']=sorted(set(new_control['quarantined'])|{old_active})
        new_control['failovers'].append(copy.deepcopy(report))
        # 本地控制指针fsync后再关闭旧主；失效旧epoch依赖这个唯一可信控制点。
        atomic_json(self.control_path,new_control)
        self.rep[old_active].shutdown()
        standby.shutdown()
        self.rep[target]=fresh
        self.control=new_control
        return report

    def audit(self):
        require(self._read_control()==self.control,'控制内存与持久状态不一致')
        self._check_artifacts()
        active=self.rep[self.control['active']]
        require(active.up,'当前活跃副本不可用')
        return {'active':self.control['active'],'epoch':self.control['epoch'],
                'ack_seq':self.control['ack_seq'], 'artifacts':self.model_identity,
                'active_audit':active.audit(),
                'standby_head':self._standby().ledger.seq}


def _replica_shutdown(self):
    for sid in list(self.pool.sessions):
        if self.pool.sessions[sid].pending is not None:self.pool.abort(sid)
        self.runner.close(sid)
    require(self.pool.audit()['live_blocks']==0,'关闭副本仍有物理KV泄漏')
    self.up=False

Replica.shutdown=_replica_shutdown


def restore_controller(root:Path) -> Coordinator:
    root=Path(root)
    tok,models=load_model_artifacts(root/'models.pt')
    control=json.loads((root/'control.json').read_text(encoding='utf-8'))
    result=Coordinator(root,tok,models,policy=control['policy'],seed=control['seed'],
                       capacity_blocks=control['capacity_blocks'],load_existing=True)
    result.audit()
    return result

In [9]:
# %% 08 保守的KV容量上界（每会话单独向上取整）
def capacity_projection(replica:Replica,available_blocks:int,reserve_blocks:int,
                        planned_new_session_lengths:list[int]|None=None):
    """每会话保守按未来最长上下文估算页数；不计共享前缀收益，避免把优化当容量保证。"""
    require(type(available_blocks) is int and available_blocks>0 and
            type(reserve_blocks) is int and reserve_blocks>=0,'容量必须非负整数且池大小为正')
    new=[] if planned_new_session_lengths is None else planned_new_session_lengths
    require(type(new) is list and all(type(n) is int and 0<n<=replica.pool.cfg.max_length for n in new),
            '新请求长度预算必须为正且不超过上下文')
    block=replica.pool.block_size
    current=0;worst=0;active=0
    for st in replica.ledger.sessions.values():
        if st.terminal is not None:continue
        active+=1
        length=len(st.prompt_ids)+len(st.generated)
        remaining=st.max_new_tokens-len(st.generated)
        require(0<=remaining<=4 and length+remaining<=replica.pool.cfg.max_length,
                '会话剩余token预算不合法')
        current+=math.ceil(length/block)
        worst+=math.ceil((length+remaining)/block)
    worst+=sum(math.ceil(x/block) for x in new)
    total=worst+reserve_blocks
    bytes_per_block=replica.pool.bytes_per_block
    return {'active_sessions':active,'current_required_blocks':current,
            'projected_session_blocks':worst,'emergency_reserved_blocks':reserve_blocks,
            'total_required_blocks':total,'available_blocks':available_blocks,
            'headroom_blocks':available_blocks-total,'allowed':available_blocks>=total,
            'bytes_per_block':bytes_per_block,'projected_reserved_bytes':total*bytes_per_block,
            'physical_capacity_bytes':available_blocks*bytes_per_block,
            'model':'per-session conservative; no prefix share credit; live KV only; excludes model weights'}

## 3. 已发布意图、实际应用和运行状态并非同一个东西

`ServingConfig` 固定字段集合：权重/模板身份、同步 WAL 模式、路由比例、租户、容量、准入、遥测比例等。

- `ConfigStore.read_active(False)` 读取发布指针与期望内容，允许下游继续检查漂移。
- `ConfigStore.detect_drift()` 比较两副本、WAL 模式及实际模型身份。
- `ConfigStore.assert_aligned()` 在读到不一致时拒绝服务；请求路由还会读取 WAL 的版本固定结果。

`observed_generation` 记录“控制器已观察过哪个期望版本”，**不代表那个版本已经健康**。`ready` 必须单独给出。状态长期未刷新，也不能凭旧的 `ready=True` 继续放流量。

In [10]:
# %% 09 声明式配置：意图、身份与运行态是三个不同事实
@dataclass(frozen=True)
class ServingConfig:
    schema: str
    model_identity_hash: str
    tokenizer_template_hash: str
    wal_policy: str
    route_canary_percent: int
    routing_salt: str
    allowed_tenants: tuple[str, ...]
    max_active_sessions: int
    reserve_blocks: int
    admission_enabled: bool
    telemetry_sample_percent: int

    def validate(self):
        require(self.schema == 'serving-config-v1', '未知配置模式，不能忽略字段')
        for label, value in (('模型身份', self.model_identity_hash), ('分词模板', self.tokenizer_template_hash)):
            require(type(value) is str and bool(re.fullmatch('[0-9a-f]{64}', value)), label+'哈希非法')
        require(self.wal_policy == 'sync', '本课零RPO发布只支持同步持久化ACK')
        require(type(self.route_canary_percent) is int and 0 <= self.route_canary_percent <= 100,
                'canary比例必须为整数0..100')
        require(type(self.telemetry_sample_percent) is int and 0 <= self.telemetry_sample_percent <= 100,
                '观测抽样比例必须为整数0..100')
        require(type(self.routing_salt) is str and bool(re.fullmatch('[A-Za-z0-9_-]{1,24}', self.routing_salt)),
                '路由salt非法')
        require(type(self.allowed_tenants) is tuple and 0 < len(self.allowed_tenants) <= 8 and
                all(type(t) is str and bool(re.fullmatch('[a-z][a-z0-9_-]{0,15}',t)) for t in self.allowed_tenants) and
                len(set(self.allowed_tenants)) == len(self.allowed_tenants) and
                tuple(sorted(self.allowed_tenants)) == self.allowed_tenants,
                '租户列表必须排序、唯一且受限')
        require(type(self.max_active_sessions) is int and 1 <= self.max_active_sessions <= 16,
                '会话上限非法')
        require(type(self.reserve_blocks) is int and 2 <= self.reserve_blocks <= 128,
                '物理KV应保留至少两页紧急余量')
        require(type(self.admission_enabled) is bool, '准入开关必须是显式bool')

    def wire(self) -> dict:
        self.validate()
        data=asdict(self);data['allowed_tenants']=list(self.allowed_tenants)
        return data

    @staticmethod
    def from_wire(data:dict):
        require(type(data) is dict and set(data)==set(ServingConfig.__dataclass_fields__),
                '配置字段不完整/有未知字段，不进行隐式降级')
        require(type(data['allowed_tenants']) is list, 'JSON中的租户必须是列表')
        obj=ServingConfig(**{**data,'allowed_tenants':tuple(data['allowed_tenants'])})
        obj.validate()
        require(obj.wire()==data,'配置无法无损往返')
        return obj


def initial_serving_config(cluster:Coordinator) -> ServingConfig:
    identity=artifact_identity(cluster.tok,cluster.models)
    result=ServingConfig('serving-config-v1',digest(identity),identity['tokenizer'],'sync',
                         0,'course83',('alpha','beta'),8,4,True,10)
    result.validate();return result


def changed_config(cfg:ServingConfig, **changes) -> ServingConfig:
    data=cfg.wire();data.update(changes)
    result=ServingConfig.from_wire(data)
    return result


def config_diff(before:ServingConfig,after:ServingConfig) -> list[dict]:
    a,b=before.wire(),after.wire()
    return [{'path':key,'old':a[key],'new':b[key],
             'severity':'critical' if key in {'model_identity_hash','tokenizer_template_hash',
               'wal_policy','allowed_tenants','max_active_sessions','reserve_blocks','admission_enabled'}
               else 'controlled'} for key in sorted(a) if a[key]!=b[key]]

In [11]:
# %% 10 单控制点版本清单、双副本预置与原子活动指针
class ChangeLedger:
    """本地hash链+单独封存head；不能证明外部主体身份或防恶意重写两者。"""
    def __init__(self,path:Path,load_existing=False):
        self.path=Path(path);self.head_path=self.path.with_suffix('.head.json')
        if load_existing:
            self.verify();return
        require(not self.path.exists() and not self.head_path.exists(),'不能覆盖历史变更审计')
        self.path.parent.mkdir(parents=True,exist_ok=True)
        self.path.touch()
        atomic_json(self.head_path,{'count':0,'head':ZERO_HASH})

    def verify(self) -> dict:
        try: lines=self.path.read_bytes().splitlines();head=json.loads(self.head_path.read_text(encoding='utf-8'))
        except (OSError,UnicodeDecodeError,json.JSONDecodeError) as exc:
            raise LessonError('变更审计缺失/损坏') from exc
        require(type(head) is dict and set(head)=={'count','head'},'审计head字段非法')
        last=ZERO_HASH;records=[]
        for i,raw in enumerate(lines,1):
            try:record=json.loads(raw.decode('utf-8'))
            except (UnicodeDecodeError,json.JSONDecodeError) as exc:raise LessonError('审计JSON行损坏') from exc
            require(type(record) is dict and set(record)=={'seq','tick','event','data','prev','hash'},
                    '变更审计字段不匹配')
            require(type(record['seq']) is int and record['seq']==i and record['prev']==last,
                    '审计事件序号/前向链错误')
            require(type(record['tick']) is int and 0<=record['tick']<=100000,
                    '逻辑tick非法')
            require(type(record['event']) is str and bool(record['event']) and
                    type(record['data']) is dict,'审计事件非法')
            core={k:v for k,v in record.items() if k!='hash'}
            require(record['hash']==digest(core),'审计内容被改写')
            last=record['hash'];records.append(record)
        require(head=={'count':len(records),'head':last},'审计内容与封存head不一致')
        return {'count':len(records),'head':last,'events':[r['event'] for r in records]}

    def add(self,tick:int,event:str,data:dict) -> dict:
        previous=self.verify()
        require(type(tick) is int and 0<=tick<=100000,'tick非法')
        require(type(event) is str and bool(re.fullmatch('[a-z_]{2,48}',event)),'事件名非法')
        require(type(data) is dict and len(canonical(data).encode())<=2048,
                '审计字段过大：不允许写入原始prompt、token和密钥')
        require(not any('prompt' in k.lower() or 'secret' in k.lower() or 'token_ids' in k.lower()
                        for k in data),'原始prompt/token不能写入审计')
        last=self.path.read_bytes().splitlines()
        if last:
            previous_record=json.loads(last[-1]);require(tick>=previous_record['tick'],'审计tick不能倒退')
        core={'seq':previous['count']+1,'tick':tick,'event':event,
              'data':copy.deepcopy(data),'prev':previous['head']}
        row={**core,'hash':digest(core)}
        with self.path.open('ab') as f:
            f.write((canonical(row)+'\n').encode('utf-8'));f.flush();os.fsync(f.fileno())
        atomic_json(self.head_path,{'count':row['seq'],'head':row['hash']})
        self.verify();return row


class ConfigStore:
    """单进程串行CAS实验，不能替代K8s apiserver/etcd/租约或真正分布式共识。"""
    def __init__(self,root:Path,cluster:Coordinator,load_existing=False):
        self.root=Path(root);self.root.mkdir(parents=True,exist_ok=True)
        self.cluster=cluster
        self.ptr=self.root/'active.json';self.versions=self.root/'versions'
        self.applied=self.root/'replicas';self.audit=ChangeLedger(self.root/'changes.jsonl',load_existing)
        self.versions.mkdir(exist_ok=True)
        for name in ('A','B'):(self.applied/name).mkdir(parents=True,exist_ok=True)
        if load_existing:
            self.read_active(require_healthy=False)
        else:
            require(not self.ptr.exists(),'活动指针不可覆盖')
            initial=initial_serving_config(cluster)
            self._stage(1,initial)
            atomic_json(self.ptr,self._pointer(1,initial))
            self.audit.add(0,'initial_release',{'generation':1,'config_sha':digest(initial.wire())})
            self.read_active(require_healthy=True)

    @staticmethod
    def _pointer(generation:int, config:ServingConfig) -> dict:
        return {'generation':generation,'sha':digest(config.wire())}

    def _version_path(self,generation:int) -> Path:
        require(type(generation) is int and generation>=1,'配置代际非法')
        return self.versions/f'{generation:06d}.json'

    def _replica_path(self,name:str,generation:int) -> Path:
        require(name in {'A','B'},'副本非法')
        return self.applied/name/f'{generation:06d}.json'

    def read_pointer(self) -> dict:
        try:state=json.loads(self.ptr.read_text(encoding='utf-8'))
        except (OSError,UnicodeDecodeError,json.JSONDecodeError) as exc:raise LessonError('活动版本指针不可读取') from exc
        require(type(state) is dict and set(state)=={'generation','sha'},'活动指针字段不对')
        require(type(state['generation']) is int and state['generation']>=1 and
                type(state['sha']) is str and bool(re.fullmatch('[0-9a-f]{64}',state['sha'])),
                '活动指针值非法')
        return state

    def _load_config(self,path:Path) -> ServingConfig:
        try:content=json.loads(path.read_text(encoding='utf-8'))
        except (OSError,UnicodeDecodeError,json.JSONDecodeError) as exc:raise LessonError('副本或版本配置损坏') from exc
        return ServingConfig.from_wire(content)

    def read_active(self,require_healthy=True) -> tuple[dict,ServingConfig]:
        p=self.read_pointer();spec=self._load_config(self._version_path(p['generation']))
        require(digest(spec.wire())==p['sha'],'活动指针与被选版本哈希不一致')
        if require_healthy:self.assert_aligned(p,spec)
        return p,spec

    def assert_aligned(self,p:dict|None=None,spec:ServingConfig|None=None) -> dict:
        if p is None or spec is None:p,spec=self.read_active(False)
        observed={}
        for name in ('A','B'):
            got=self._load_config(self._replica_path(name,p['generation']))
            require(got.wire()==spec.wire(),f'{name}副本配置漂移；停止新请求与自动接管')
            observed[name]=digest(got.wire())
        model_identity=artifact_identity(self.cluster.tok,self.cluster.models)
        require(spec.model_identity_hash==digest(model_identity) and
                spec.tokenizer_template_hash==model_identity['tokenizer'],
                '运行模型或模板身份与配置不一致')
        require(self.cluster.control['policy']==spec.wal_policy,'WAL确认模式发生漂移')
        require(self.cluster._read_control()==self.cluster.control,'控制指针与内存不一致')
        require(self.cluster.model_identity==model_identity,'实际模型权重身份变化')
        require(self.cluster.capacity_blocks>=spec.reserve_blocks,'物理KV页容量不足')
        return {'generation':p['generation'],'sha':p['sha'],'replicas':observed,
                'epoch':self.cluster.control['epoch'],'model_identity_hash':digest(model_identity)}

    def _immutable_json(self,path:Path,content:dict):
        if path.exists():
            existing=self._load_config(path)
            require(existing.wire()==content,'预置文件已存在但内容不同：不能自动覆盖')
        else:
            atomic_json(path,content)

    def _stage(self,generation:int,cfg:ServingConfig,fail_after:str|None=None):
        data=cfg.wire()
        self._immutable_json(self._version_path(generation),data)
        for name in ('A','B'):
            self._immutable_json(self._replica_path(name,generation),data)
            if fail_after=='stage_'+name:
                raise SimulatedCrash('只预置了部分副本：活动指针未改变，不能宣称发布成功')
        # 预置阶段的文件只能通过指针切换变为生效配置。
        for name in ('A','B'):
            require(self._load_config(self._replica_path(name,generation)).wire()==data,
                    '预置副本内容不匹配')

    def snapshot(self) -> dict:
        p,s=self.read_active(require_healthy=True)
        control=self.cluster.control
        return {'pointer':copy.deepcopy(p),'config_sha':digest(s.wire()),
                'epoch':control['epoch'],'active':control['active'],
                'ack_seq':control['ack_seq'],'control_sha':digest(control),
                'A_seq':self.cluster.rep['A'].ledger.seq,
                'B_seq':self.cluster.rep['B'].ledger.seq,
                'artifact_sha':digest(self.cluster.model_identity)}

    def probe_kv(self) -> dict:
        self.assert_aligned()
        reports={};max_delta=0.0
        tok=self.cluster.tok
        prompt=tok.apply_chat_template([{'role':'user','content':'hello'}],
                                       tokenize=True,add_generation_prompt=True)
        for name in ('A','B'):
            replica=self.cluster.rep[name]
            require(replica.up,'副本未运行，发布预检拒绝')
            require(artifact_identity(tok,replica.models)==self.cluster.model_identity,
                    '副本实际模型权重身份与发布不匹配')
            for version in ('stable','canary'):
                pool=PagedKVPool(32,4,next(iter(replica.models.values())).cfg,set(replica.models))
                runner=CachedModelRunner(tok,replica.models,pool)
                sid='preflight_'+name+'_'+version
                runner.open_prefill(sid,version,prompt)
                delta=runner.assert_consistent(sid)
                max_delta=max(max_delta,delta)
                runner.close(sid)
                require(pool.audit()['live_blocks']==0,'preflight的物理KV没有释放')
                reports[name+'_'+version]={'max_cache_full_diff':delta,'released':True}
        return {'max_cache_full_diff':max_delta,'checked':reports,'real_tensor_kv':True}

    def safe_capacity(self,proposed:ServingConfig,planned_new_session_lengths:list[int]|None=None) -> dict:
        proposed.validate()
        result={}
        for name in ('A','B'):
            replica=self.cluster.rep[name]
            check=capacity_projection(replica,self.cluster.capacity_blocks,proposed.reserve_blocks,
                                      planned_new_session_lengths)
            require(check['allowed'],f'{name}可恢复KV容量不够；不准上线')
            require(check['active_sessions']<=proposed.max_active_sessions,
                    f'{name}现有会话数超过新配置上限')
            result[name]=check
        return result

    def preflight(self,candidate:ServingConfig,tick:int,ttl_ticks:int=4) -> dict:
        require(type(tick) is int and tick>=0 and type(ttl_ticks) is int and 1<=ttl_ticks<=20,
                '使用非负的教学逻辑tick；TTL为1..20')
        candidate.validate();p,old=self.read_active(require_healthy=True)
        require(candidate.model_identity_hash==old.model_identity_hash and
                candidate.tokenizer_template_hash==old.tokenizer_template_hash and
                candidate.wal_policy==old.wal_policy,
                '模型/模板/WAL身份变更需要独立的新artifact迁移，不能只修改配置字段')
        for name in ('A','B'):
            for st in self.cluster.rep[name].ledger.sessions.values():
                require(st.terminal is not None or st.tenant in candidate.allowed_tenants,
                        '存在活跃会话时不能撤销该租户访问权限')
        checks=self.safe_capacity(candidate)
        probes=self.probe_kv()
        snap=self.snapshot()
        report={'kind':'release_preflight','base':snap,'candidate':candidate.wire(),
                'candidate_sha':digest(candidate.wire()),'issued_tick':tick,'expires_tick':tick+ttl_ticks,
                'capacity':checks,'probe':probes,'diff':config_diff(old,candidate),
                'decision':'allow','limitations':'logical_ticks_single_controller_no_authenticated_approval'}
        report['seal']=digest(report)
        return report

    def verify_preflight(self,proof:dict,candidate:ServingConfig,tick:int) -> None:
        require(type(proof) is dict and 'seal' in proof and
                proof['seal']==digest({k:v for k,v in proof.items() if k!='seal'}),
                '发布证据完整性错误')
        require(proof['kind']=='release_preflight' and proof['decision']=='allow',
                '证据没有明确放行')
        require(type(tick) is int and proof['issued_tick']<=tick<=proof['expires_tick'],
                '预检证据尚未生效或已超过逻辑TTL')
        require(proof['candidate']==candidate.wire() and proof['candidate_sha']==digest(candidate.wire()),
                '预检证据绑定的是另一个候选')
        require(proof['base']==self.snapshot(),
                '预检后控制epoch/ACK/副本状态/生效配置改变，必须重新取证')
        self.safe_capacity(candidate)
        self.probe_kv()

    @staticmethod
    def approval(candidate:ServingConfig,base_generation:int,actor='local_operator') -> dict:
        return {'actor':actor,'ticket':'CHG-0083','approved':True,
                'candidate_sha':digest(candidate.wire()),'expected_generation':base_generation}

    def _approval(self,approve:dict,candidate:ServingConfig,expected_generation:int):
        require(type(approve) is dict and set(approve)==
                {'actor','ticket','approved','candidate_sha','expected_generation'},
                '审批记录字段不匹配')
        require(approve['approved'] is True and approve['expected_generation']==expected_generation and
                approve['candidate_sha']==digest(candidate.wire()) and
                type(approve['actor']) is str and bool(approve['actor']) and
                approve['ticket']=='CHG-0083','模拟审批没有绑定这次变更或代际')

    def publish(self,candidate:ServingConfig,proof:dict,approval:dict,
                expected_generation:int,tick:int,fail_after:str|None=None) -> dict:
        p,_=self.read_active(require_healthy=True)
        require(type(expected_generation) is int and p['generation']==expected_generation,
                '配置CAS预期代际过期；拒绝并行覆盖')
        self._approval(approval,candidate,expected_generation)
        self.verify_preflight(proof,candidate,tick)
        generation=expected_generation+1
        self._stage(generation,candidate,fail_after)
        # 提交前再次检查；这只是单进程串行CAS模型，跨进程仍需真正的控制服务。
        require(self.read_pointer()==p,'预置期间活动指针变化，停止提交')
        self.verify_preflight(proof,candidate,tick)
        atomic_json(self.ptr,self._pointer(generation,candidate))
        if fail_after=='after_commit':
            raise SimulatedCrash('活动指针已切换但客户端未收到ACK：必须读取实际版本，不允许自动再次发布')
        self.assert_aligned()
        self.audit.add(tick,'release_committed',{'generation':generation,'candidate_sha':digest(candidate.wire()),
                                                'from_generation':expected_generation})
        return {'generation':generation,'active_sha':digest(candidate.wire()),
                'diff':proof['diff'],'ack':'committed','cas_expected':expected_generation}

    def detect_drift(self) -> dict:
        p,s=self.read_active(False)
        findings=[]
        for name in ('A','B'):
            try:observed=self._load_config(self._replica_path(name,p['generation']))
            except LessonError as exc:
                findings.append({'replica':name,'status':'invalid','reason':str(exc)})
                continue
            for change in config_diff(s,observed):
                findings.append({'replica':name,'status':'drift',**change})
        actual=artifact_identity(self.cluster.tok,self.cluster.models)
        if digest(actual)!=s.model_identity_hash:
            findings.append({'replica':'models','status':'drift','path':'model_identity_hash'})
        if self.cluster.control['policy']!=s.wal_policy:
            findings.append({'replica':'wal','status':'drift','path':'wal_policy'})
        return {'generation':p['generation'],'findings':findings,'healthy':not findings}

    def reconcile(self,name:str,approval:dict,tick:int,expected_generation:int):
        require(name in {'A','B'},'副本名非法')
        p,s=self.read_active(False)
        require(p['generation']==expected_generation,'修复预期代际过期')
        self._approval(approval,s,expected_generation)
        self._load_config(self._version_path(p['generation']))
        # 单控制者的显式修复动作，不掩盖WAL/真实权重漂移，不自动删除历史。
        path=self._replica_path(name,p['generation'])
        before=self.detect_drift()['findings']
        require(any(item['replica']==name for item in before),'未检测到该副本漂移，拒绝无意义覆盖')
        atomic_json(path,s.wire())
        self.assert_aligned()
        self.audit.add(tick,'drift_reconciled',{'generation':expected_generation,
                                                'replica':name,'restored_sha':digest(s.wire())})
        return {'reconciled':name,'generation':expected_generation,'healthy':True}

    def _has_commit_receipt(self,generation:int,sha:str) -> bool:
        self.audit.verify()
        for line in self.audit.path.read_text(encoding='utf-8').splitlines():
            record=json.loads(line)
            data=record['data']
            if record['event'] in {'initial_release','release_committed','release_reconciled'} and \
               data.get('generation')==generation and data.get('candidate_sha',data.get('config_sha'))==sha:
                return True
        return False

    def resolve_uncertain_commit(self,candidate:ServingConfig,proof:dict,approval:dict,
                                 expected_generation:int,tick:int) -> dict:
        """指针已经提交而ACK/审计未完成：读取真实指针，不盲目再递增版本。"""
        self._approval(approval,candidate,expected_generation)
        require(type(proof) is dict and proof.get('seal')==digest({k:v for k,v in proof.items() if k!='seal'}),
                '无法核对原始预检证据')
        require(proof.get('candidate_sha')==digest(candidate.wire()) and
                proof.get('base',{}).get('pointer',{}).get('generation')==expected_generation,
                '含糊ACK绑定其他候选或前驱代际')
        require(type(tick) is int and proof['issued_tick']<=tick<=proof['expires_tick'],
                '含糊ACK过久，需要重新建立人工处置流程')
        p,s=self.read_active(require_healthy=True)
        require(p==self._pointer(expected_generation+1,candidate) and s.wire()==candidate.wire(),
                '未找到与原提交完全一致的活动指针；不能假装已经发布')
        require(self.cluster.control['epoch']==proof['base']['epoch'] and
                self.cluster.control['ack_seq']==proof['base']['ack_seq'] and
                self.cluster.control['active']==proof['base']['active'],
                '提交后出现新流量或切主，不能直接补写旧审计')
        recorded=self._has_commit_receipt(p['generation'],p['sha'])
        if not recorded:
            self.audit.add(tick,'release_reconciled',{'generation':p['generation'],
                              'candidate_sha':p['sha'],'from_generation':expected_generation})
        return {'generation':p['generation'],'committed':True,'audit_reconciled':not recorded,
                'pointer_advanced_again':False}

    def route(self,tenant:str,key:str,text:str='hello',max_new_tokens:int=4) -> dict:
        p,s=self.read_active(require_healthy=True)
        require(self._has_commit_receipt(p['generation'],p['sha']),
                '活动配置尚未有完整提交审计，先核对不确定ACK')
        require(s.admission_enabled,'当前配置停止接受新请求')
        require(tenant in s.allowed_tenants,'租户不在授权列表')
        active=self.cluster.rep[self.cluster.control['active']]
        require(active.up,'活动实例不可用')
        rid=make_rid(tenant,key)
        if rid in active.ledger.sessions:
            version=active.ledger.sessions[rid].version
        else:
            current=sum(st.terminal is None for st in active.ledger.sessions.values())
            require(current<s.max_active_sessions,'达到活动会话上限')
            future_prompt=self.cluster.tok.apply_chat_template(
                [{'role':'user','content':text}],tokenize=True,add_generation_prompt=True)
            self.safe_capacity(s,[len(future_prompt)+max_new_tokens])
            bucket=int(digest({'tenant':tenant,'key':key,'salt':s.routing_salt})[:12],16)%100
            version='canary' if bucket<s.route_canary_percent else 'stable'
        receipt=self.cluster.start(self.cluster.control['epoch'],tenant,key,version,text,max_new_tokens)
        return {**receipt,'version':version,'config_generation':p['generation'],
                'config_sha':p['sha'],'pinned_to_wal':True}

    def step(self,rid:str) -> dict:
        p,_=self.read_active(require_healthy=True)
        require(self._has_commit_receipt(p['generation'],p['sha']),
                '活动配置缺少提交审计，停止生成')
        active=self.cluster.rep[self.cluster.control['active']]
        require(rid in active.ledger.sessions,'会话不存在')
        st=active.ledger.sessions[rid]
        require(st.terminal is None,'会话已经结束')
        return self.cluster.step(self.cluster.control['epoch'],rid,len(st.generated))

In [12]:
# %% 11 故障接管的发布配置校验和端到端KV验收
class RecoveryGate:
    def __init__(self,store:ConfigStore):self.store=store

    def preflight(self,target:str,tick:int,ttl_ticks:int=4) -> dict:
        store=self.store;cluster=store.cluster
        require(target in {'A','B'} and target!=cluster.control['active'],'目标应是另一个副本')
        require(type(tick) is int and tick>=0 and type(ttl_ticks) is int and 1<=ttl_ticks<=20,
                '逻辑时间/TTL非法')
        p,s=store.read_active(require_healthy=True)
        require(s.wal_policy=='sync','本课端到端零RPO验收要求sync')
        standby=cluster.rep[target]
        require(standby.up,'接管目标不可用')
        require(standby.ledger.seq>=cluster.control['ack_seq'],
                '目标没有全部已ACK日志，不能无损接管')
        require(standby.journal.records==cluster.rep[cluster.control['active']].journal.records[:len(standby.journal.records)],
                '接管目标日志不是当前主的前缀')
        capacity=store.safe_capacity(s)[target]
        audit=standby.audit();require(audit['max_cache_full_logits_diff']<=2e-5,
                                      '真实KV与全上下文输出不一致')
        payload={'kind':'recovery_preflight','generation':p['generation'],
                 'config_sha':p['sha'],'epoch':cluster.control['epoch'],
                 'old_active':cluster.control['active'],'target':target,
                 'ack_seq':cluster.control['ack_seq'],'target_seq':standby.ledger.seq,
                 'target_head':standby.ledger.last_hash,'control_sha':digest(cluster.control),
                 'issued_tick':tick,'expires_tick':tick+ttl_ticks,'capacity':capacity,
                 'target_kv_max_abs_diff':audit['max_cache_full_logits_diff'],
                 'target_active_sessions':audit['active_sessions'],
                 'estimated_rto_logical_ticks':2+math.ceil(len(standby.journal.records)/16)+
                    math.ceil(sum(len(st.prompt_ids)+len(st.generated) for st in standby.ledger.sessions.values()
                                  if st.terminal is None)/32),
                 'decision':'allow'}
        payload['seal']=digest(payload)
        return payload

    def promote(self,proof:dict,tick:int,rto_budget_ticks:int=40) -> dict:
        store=self.store;cluster=store.cluster
        require(type(proof) is dict and proof.get('seal')==digest({k:v for k,v in proof.items() if k!='seal'}),
                '恢复门槛证据被修改')
        require(type(tick) is int and proof['issued_tick']<=tick<=proof['expires_tick'],
                '恢复证据已过期')
        require(proof['decision']=='allow','恢复证据未放行')
        p,s=store.read_active(require_healthy=True)
        require(p['generation']==proof['generation'] and p['sha']==proof['config_sha'],
                '恢复前配置代际被改动')
        require(cluster.control['epoch']==proof['epoch'] and
                cluster.control['active']==proof['old_active'] and
                cluster.control['ack_seq']==proof['ack_seq'] and
                digest(cluster.control)==proof['control_sha'],
                '恢复前控制epoch/ACK发生变化，必须重新取证')
        target=proof['target']
        require(cluster.rep[target].ledger.seq==proof['target_seq'] and
                cluster.rep[target].ledger.last_hash==proof['target_head'],
                '恢复目标日志被改动')
        store.safe_capacity(s)
        require(type(rto_budget_ticks) is int and rto_budget_ticks>=proof['estimated_rto_logical_ticks'],
                '恢复逻辑RTO预算过小；不允许先关闭健康主机再失败')
        # 明确模拟主机宕机，但并不接触真实主机或外部网络。
        cluster.rep[cluster.control['active']].shutdown()
        result=cluster.promote(proof['epoch'],target,rpo_budget_ticks=0,
                               rto_budget_ticks=rto_budget_ticks)
        require(result['lost_acked_events']==0 and result['rpo_gap_logical_ticks']==0,
                '切主结果不满足零RPO')
        require(cluster.control['epoch']==proof['epoch']+1 and
                cluster.control['active']==target,'旧主没有被epoch隔离')
        store.assert_aligned()
        audit=cluster.rep[target].audit()
        require(audit['max_cache_full_logits_diff']<=2e-5,'重新计算的KV与全前向不一致')
        check={'failover':result,'active':target,'epoch':cluster.control['epoch'],
               'old_epoch_fenced':True,'actual_kv_rebuilt':True,
               'max_kv_full_logits_diff':audit['max_cache_full_logits_diff'],
               'active_sessions_rebuilt':audit['active_sessions'],
               'config_generation_preserved':store.read_pointer()['generation']==proof['generation'],
               'zero_rpo_verified':True,'logical_time_only':True}
        store.audit.add(tick,'recovery_accepted',{'generation':proof['generation'],
                                                'epoch':cluster.control['epoch'],'ack_seq':cluster.control['ack_seq']})
        return check

In [13]:
print('发布配置字段：', list(ServingConfig.__dataclass_fields__))
print('ConfigStore 的数据来自本地 generation 指针及双副本文件。')

发布配置字段： ['schema', 'model_identity_hash', 'tokenizer_template_hash', 'wal_policy', 'route_canary_percent', 'routing_salt', 'allowed_tenants', 'max_active_sessions', 'reserve_blocks', 'admission_enabled', 'telemetry_sample_percent']
ConfigStore 的数据来自本地 generation 指针及双副本文件。


## 4. 安全白名单：按“风险”分类，而不是只按字段变化大小分类

本课固定：一个副本、一个字段、状态有效，且字段必须是 `telemetry_sample_percent`，才允许自动修复。该字段也不一定在所有真实系统里低风险；这里是明确的教学授权假设。

- `allowed_tenants`：可能撤销或授予访问，自动覆盖有权限风险。
- `route_canary_percent`：即使 schema 标为 `controlled`，也会改变用户流量，因此**不自动修复**。
- `max_active_sessions`、`reserve_blocks`：影响恢复容量与 admission，必须停在人工门槛。
- 模型身份、WAL、未知字段、坏 JSON：不能从“期望值”强行覆盖无法解释的实际异常。
- 双副本同时漂移：本课的单点修复器不能保证一次操作恢复整体健康，因此升级人工。

注意：**先判断是否允许动作，再决定是修复、宽限还是退避。** 不应在意外修改发生后才补审计。

In [14]:
# %% 12 声明式调谐策略：自动修复范围必须事先收窄
@dataclass(frozen=True)
class ReconcilePolicy:
    auto_paths: tuple[str,...] = ('telemetry_sample_percent',)
    grace_ticks: int = 2
    retry_base_ticks: int = 2
    retry_cap_ticks: int = 8
    max_retries: int = 3
    status_fresh_ticks: int = 2

    def validate(self):
        # 课程明确授权的唯一低风险字段；即使调用者改配置，也不能扩大自动更改权限。
        require(type(self.auto_paths) is tuple and self.auto_paths == ('telemetry_sample_percent',),
                '本课自动修复权限仅限telemetry_sample_percent，不能把路由/租户/容量加入')
        for name in ('grace_ticks','retry_base_ticks','retry_cap_ticks','max_retries','status_fresh_ticks'):
            v=getattr(self,name)
            require(type(v) is int and v>=1, name+'必须为正整数')
        require(self.retry_cap_ticks>=self.retry_base_ticks,'最大退避不能小于基础退避')


def classify_drift(findings:list[dict],policy:ReconcilePolicy) -> dict:
    """分类时严格白名单，未知/损坏/跨副本混合一律人工处理。"""
    policy.validate()
    require(type(findings) is list,'漂移需要列表')
    if not findings:
        return {'mode':'healthy','replica':None,'paths':[]}
    paths=[]
    for finding in findings:
        require(type(finding) is dict,'非法漂移记录')
        paths.append(f"{finding.get('replica','?')}.{finding.get('path',finding.get('status','?'))}")
    if len(findings)==1:
        item=findings[0]
        if (item.get('status')=='drift' and item.get('replica') in ('A','B')
                and item.get('path') in policy.auto_paths
                and item.get('severity')=='controlled'):
            return {'mode':'auto_safe','replica':item['replica'],'paths':paths}
    return {'mode':'manual','replica':None,'paths':sorted(paths)}

In [15]:
policy=ReconcilePolicy()
print('低风险单处:',classify_drift([{'replica':'B','path':'telemetry_sample_percent',
    'status':'drift','severity':'controlled'}],policy))
for path,severity in [('route_canary_percent','controlled'),('allowed_tenants','critical')]:
    print(path,'→',classify_drift([{'replica':'B','path':path,
        'status':'drift','severity':severity}],policy)['mode'])

低风险单处: {'mode': 'auto_safe', 'replica': 'B', 'paths': ['B.telemetry_sample_percent']}
route_canary_percent → manual
allowed_tenants → manual


## 5. Python 基础提示：`@dataclass`、字典、异常与快照

`@dataclass(frozen=True)` 帮我们把调谐权限固定在构造时；`.validate()` 仍负责严格检查范围。它不是生产鉴权，**拿到源码的人当然能写另一个控制器**。

- `status['ready']` 取字典字段；`state.update(...)` 同时替换若干字段。
- `copy.deepcopy()` 避免把“先前观察”与当前字典放在同一个对象里。
- `raise LessonError(...)` 在条件不满足时**拒绝**，不把故障假装成成功。
- SHA256 封印用于发现意外写坏的状态文件；没有外部可信签名与写权限保护时，它无法证明操作者身份。
- `type(value) is int` 有意排除 Python 中也属于整数子类的 `True/False`。

观察与实际修复分开，是为了在本课直接测试“宽限期不动、失败时退避、危险漂移不动”。

## 6. 单进程 lease 与 epoch：过期控制者不能继续写

两个控制器都看到同一份期望时，旧控制者不应该继续在新控制者接管后修复配置。因此本课把本地 lease 绑定到 `holder`、`term`、`cluster_epoch` 与 `expires_tick`。

同持有者且未过期的重新获取会返回同一份 lease；新持有者只能在旧租约过期或 epoch 更迭后取得新 term。每次调谐前都检查当前文件和 epoch，陈旧凭证拒绝。

**边界：** `atomic_json()` 与本地令牌只用于单进程串行教学；它们不是 Kubernetes Lease、etcd compare-and-swap、分布式选主、真实身份认证或并发互斥证明。

In [16]:
# %% 13 单进程教学Lease与epoch围栏；不模拟外部共识
class LocalLease:
    """本地教学CAS/租约围栏；不是跨进程互斥、鉴权或K8s leader election。"""
    def __init__(self,path:Path,cluster:Coordinator):
        self.path=Path(path);self.path.parent.mkdir(parents=True,exist_ok=True)
        self.cluster=cluster

    def read(self):
        if not self.path.exists():return None
        try:raw=json.loads(self.path.read_text(encoding='utf-8'))
        except (OSError,UnicodeDecodeError,json.JSONDecodeError) as exc:
            raise LessonError('本地lease损坏，停止调谐') from exc
        require(type(raw) is dict and set(raw)=={'holder','term','cluster_epoch','issued_tick',
                    'expires_tick','version','token','seal'},'lease字段非法')
        require(raw['version']==VERSION and raw['seal']==digest({k:v for k,v in raw.items() if k!='seal'}),
                'lease完整性错误')
        require(type(raw['holder']) is str and bool(re.fullmatch('[a-zA-Z0-9_-]{1,24}',raw['holder'])),
                'lease持有者非法')
        require(all(type(raw[k]) is int and raw[k]>=0 for k in ('term','cluster_epoch','issued_tick','expires_tick')),
                'lease计数非法')
        require(raw['term']>=1 and raw['cluster_epoch']>=1 and raw['expires_tick']>=raw['issued_tick'],
                'lease时间非法')
        return raw

    def acquire(self,holder:str,tick:int,ttl:int=20) -> dict:
        bounded_str(holder,24,'holder')
        require(bool(re.fullmatch('[a-zA-Z0-9_-]{1,24}',holder)), 'holder字符非法')
        require(type(tick) is int and tick>=0 and type(ttl) is int and 2<=ttl<=100,
                '教学租约tick/ttl非法')
        current_epoch=self.cluster._read_control()['epoch']
        old=self.read()
        if old and old['expires_tick']>=tick and old['cluster_epoch']==current_epoch:
            require(old['holder']==holder,'当前lease由另一控制者持有；旧控制者不得越权')
            return copy.deepcopy(old)
        term=old['term']+1 if old else 1
        core={'holder':holder,'term':term,'cluster_epoch':current_epoch,
              'issued_tick':tick,'expires_tick':tick+ttl,'version':VERSION,
              'token':digest({'holder':holder,'term':term,'epoch':current_epoch,'issued':tick})}
        value={**core,'seal':digest(core)}
        atomic_json(self.path,value)
        return value

    def verify(self,lease:dict,tick:int):
        require(type(tick) is int and tick>=0,'tick非法')
        current=self.read()
        require(current is not None and type(lease) is dict and lease==current,
                '控制者lease过期/不属于当前控制者')
        require(current['issued_tick']<=tick<=current['expires_tick'],
                '控制者lease超过有效tick')
        require(current['cluster_epoch']==self.cluster._read_control()['epoch']
                and current['cluster_epoch']==self.cluster.control['epoch'],
                '控制epoch发生切换，必须重新选举/取lease')
        return True

## 7. Status 条件：Alive 不等于 Ready

本课的状态持久化在 `status.json`，包含：

| 字段 | 作用 |
|---|---|
| `observed_generation` / `desired_sha` | 本次检查绑定了哪个真实活动版本 |
| `cluster_epoch` | 避免拿旧主结果指挥新主 |
| `ready` / `live` | 新请求准入与控制程序存活区分 |
| `reason` | `Ready`、`GracePeriod`、`RetryBackoff`、`Quarantined`、`ManualReview`、`AutoRepaired` |
| `first_seen_tick` | 记录同一漂移第一次观察时间 |
| `retry_count` / `next_retry_tick` | 有界退避，不空转重试 |
| `last_tick` / `last_success_tick` | 检测状态是否过期 |
| `finding_sha` | 只对**同一代际、同一漂移证据**累计宽限与重试 |

`ManualReview`/`Quarantined` 时 `live=True` 可以帮助诊断，但 `ready=False` 必须拒绝新请求。旧 status 的 `ready=True` 也不能绕过新的配置指针或模型身份核对。

## 8. Reconcile 主循环：反复读实际，不执行旧命令队列

主循环 `reconcile_once(tick, lease)`：

1. 先核对 lease、epoch 和逻辑时钟，**同 tick 重复事件立即返回且不增加修复次数**。
2. 从磁盘读期望版本、实际副本配置、WAL控制指针和模型权重身份。
3. 分类漂移。健康则 `Ready`；危险或损坏立即 `ManualReview` 并阻断。
4. 对许可的单处遥测漂移，先等待两个完整 tick；仍相同才允许尝试。
5. 修复前再次核对真实双副本容量与 WAL；成功后重新观察。
6. 暂时性 I/O 错误退避 **2、4、8 tick**，失败至上限改为 `Quarantined`，不会无限空转。

*自动调谐* 不会自动发布新 `generation`，也不删除 WAL 和模型工件。异常分类决定是否进入人工处理，不把所有错误都作为“稍后再试”的无限重试。操作员因此能看到明确的状态原因。[Operator 错误处理讨论](https://www.redhat.com/en/blog/kubernetes-operators-best-practices)

In [17]:
# %% 14 控制器Status、去重调谐、退避、危险漂移阻断
class ReconcileLoop:
    """Level-based单步调谐。函数每次都读取磁盘真实期望与观测，而非执行排队命令。

    本地状态/指纹不是身份签名，控制者只模拟单进程串行；严禁与真实集群控制混淆。
    """
    STATUS_FIELDS={'version','observed_generation','desired_sha','cluster_epoch','ready','live',
                   'reason','mode','finding_sha','first_seen_tick','retry_count','next_retry_tick',
                   'last_tick','last_success_tick','last_error','quarantined','paths'}

    def __init__(self,root:Path,store:ConfigStore,policy:ReconcilePolicy|None=None,
                 load_existing=False):
        self.root=Path(root);self.root.mkdir(parents=True,exist_ok=True)
        self.store=store;self.policy=policy or ReconcilePolicy();self.policy.validate()
        self.status_path=self.root/'status.json'
        self.lease=LocalLease(self.root/'lease.json',store.cluster)
        self.events=ChangeLedger(self.root/'reconcile_events.jsonl',load_existing)
        self.inject_remaining_failures=0
        if load_existing:
            self.read_status()
        else:
            require(not self.status_path.exists(),'调谐状态不能被覆盖')
            initial={'version':VERSION,'observed_generation':0,'desired_sha':'',
                     'cluster_epoch':store.cluster.control['epoch'],'ready':False,'live':True,
                     'reason':'Bootstrapping','mode':'unknown','finding_sha':'',
                     'first_seen_tick':None,'retry_count':0,'next_retry_tick':0,
                     'last_tick':-1,'last_success_tick':None,'last_error':'',
                     'quarantined':False,'paths':[]}
            self._save_status(initial)

    def read_status(self) -> dict:
        try:blob=json.loads(self.status_path.read_text(encoding='utf-8'))
        except (OSError,UnicodeDecodeError,json.JSONDecodeError) as exc:
            raise LessonError('状态文件缺失/损坏，必须fail-closed') from exc
        require(type(blob) is dict and set(blob)=={'status','seal'},'status封装非法')
        st=blob['status']
        require(type(st) is dict and set(st)==self.STATUS_FIELDS and blob['seal']==digest(st),
                '状态字段/摘要不匹配')
        require(st['version']==VERSION and type(st['observed_generation']) is int
                and st['observed_generation']>=0 and type(st['cluster_epoch']) is int
                and st['cluster_epoch']>=1,'状态版本或代际非法')
        require(all(type(st[k]) is bool for k in ('ready','live','quarantined')),
                '状态布尔字段非法')
        require(type(st['retry_count']) is int and st['retry_count']>=0
                and type(st['next_retry_tick']) is int and st['next_retry_tick']>=0
                and type(st['last_tick']) is int and st['last_tick']>=-1,'状态时钟/计数非法')
        require(st['first_seen_tick'] is None or
                (type(st['first_seen_tick']) is int and st['first_seen_tick']>=0),
                '漂移首次时间非法')
        require(st['last_success_tick'] is None or
                (type(st['last_success_tick']) is int and st['last_success_tick']>=0),
                '健康时间非法')
        require(type(st['paths']) is list and all(type(x) is str for x in st['paths']),
                '漂移字段列表非法')
        return st

    def _save_status(self,value:dict) -> None:
        require(set(value)==self.STATUS_FIELDS,'内部状态字段不完整')
        atomic_json(self.status_path,{'status':value,'seal':digest(value)})
        self.read_status()

    def _finish(self,previous,st,tick,action='observe'):
        st['last_tick']=tick
        self._save_status(st)
        changed=(st['reason']!=previous['reason'] or st['ready']!=previous['ready']
                 or st['observed_generation']!=previous['observed_generation'])
        if changed or action!='observe':
            self.events.add(tick,'reconcile_decision',{'generation':st['observed_generation'],
                'reason':st['reason'],'mode':st['mode'],'ready':st['ready'],
                'action':action,'finding_sha':st['finding_sha'][:16]})
        return copy.deepcopy(st)

    def reconcile_once(self,tick:int,lease:dict,crash_after_repair=False):
        self.lease.verify(lease,tick)
        prev=self.read_status()
        require(type(tick) is int and tick>=prev['last_tick'],'不能倒退逻辑tick')
        if tick==prev['last_tick']:
            return copy.deepcopy(prev)   # 同tick重复事件完全幂等；但仍核对lease。
        st=copy.deepcopy(prev)
        st['ready']=False;st['live']=True
        try:
            # 使用真实指针、两副本配置、模型权重身份与WAL控制指针。
            pointer,desired=self.store.read_active(require_healthy=False)
            require(self.store.cluster._read_control()==self.store.cluster.control,
                    '内存控制状态与持久epoch/ACK不一致')
            self.store.cluster._check_artifacts()
            report=self.store.detect_drift()
            classification=classify_drift(report['findings'],self.policy)
        except (LessonError,OSError) as exc:
            st.update(reason='ManualReview',mode='manual',paths=['untrusted_control_or_artifact'],
                      last_error=str(exc),quarantined=True,finding_sha='',first_seen_tick=tick,
                      retry_count=0,next_retry_tick=0)
            return self._finish(prev,st,tick,'block')
        basis=digest({'generation':pointer['generation'],'sha':pointer['sha'],
                      'epoch':self.store.cluster.control['epoch'],'findings':report['findings']})
        if (st['finding_sha']!=basis or st['observed_generation']!=pointer['generation']
                or st['cluster_epoch']!=self.store.cluster.control['epoch']):
            st.update(first_seen_tick=tick,retry_count=0,next_retry_tick=0,quarantined=False)
        st.update(observed_generation=pointer['generation'],desired_sha=pointer['sha'],
                  cluster_epoch=self.store.cluster.control['epoch'],finding_sha=basis,
                  paths=classification['paths'],mode=classification['mode'],last_error='')
        if classification['mode']=='healthy':
            try:self.store.assert_aligned(pointer,desired)
            except LessonError as exc:
                st.update(reason='ManualReview',mode='manual',ready=False,quarantined=True,
                          last_error=str(exc),paths=['untrusted_runtime_state'])
                return self._finish(prev,st,tick,'block')
            st.update(reason='Ready',ready=True,quarantined=False,first_seen_tick=None,
                      retry_count=0,next_retry_tick=0,last_success_tick=tick,paths=[])
            return self._finish(prev,st,tick)
        if classification['mode']=='manual':
            st.update(reason='ManualReview',ready=False,quarantined=True)
            return self._finish(prev,st,tick,'block')
        # 仅一处低风险遥测比例漂移才可进入自动修复，其余全部fail-closed。
        if st['quarantined']:
            st['reason']='Quarantined'
            return self._finish(prev,st,tick)
        if tick-st['first_seen_tick']<self.policy.grace_ticks:
            st['reason']='GracePeriod'
            return self._finish(prev,st,tick)
        if tick<st['next_retry_tick']:
            st['reason']='RetryBackoff'
            return self._finish(prev,st,tick)
        try:
            # 不授权改变模型、WAL、租户或容量；修复前核对两副本容量与权重。
            self.store.safe_capacity(desired)
            self.store.cluster.audit()
            if self.inject_remaining_failures>0:
                self.inject_remaining_failures-=1
                raise SimulatedTransientRepair('本地模拟临时I/O错误，未触碰配置')
            self.store.reconcile(classification['replica'],
                                 self.store.approval(desired,pointer['generation']),
                                 tick,pointer['generation'])
            if crash_after_repair:
                raise SimulatedCrash('模拟修复已落盘，但状态尚未提交：恢复后应重新观察')
            require(self.store.detect_drift()['healthy'],'修复后仍有漂移')
        except SimulatedTransientRepair as exc:
            st['retry_count']+=1
            if st['retry_count']>=self.policy.max_retries:
                st.update(reason='Quarantined',quarantined=True,next_retry_tick=0,
                          last_error=str(exc))
                return self._finish(prev,st,tick,'quarantine')
            wait=min(self.policy.retry_cap_ticks,self.policy.retry_base_ticks*2**(st['retry_count']-1))
            st.update(reason='RetryBackoff',next_retry_tick=tick+wait,last_error=str(exc))
            return self._finish(prev,st,tick,'backoff')
        except LessonError as exc:
            st.update(reason='ManualReview',mode='manual',ready=False,quarantined=True,
                      last_error=str(exc))
            return self._finish(prev,st,tick,'block')
        st.update(reason='AutoRepaired',ready=True,quarantined=False,first_seen_tick=None,
                  retry_count=0,next_retry_tick=0,last_success_tick=tick,paths=[])
        return self._finish(prev,st,tick,'auto_repair')

    def route(self,tick:int,tenant:str,key:str,text='hello',max_new_tokens=4) -> dict:
        st=self.read_status();p,s=self.store.read_active(require_healthy=False)
        require(type(tick) is int and tick>=0,'路由逻辑tick非法')
        require(st['ready'] and st['live'] and not st['quarantined'],
                '控制循环未宣告Ready，拒绝新请求')
        require(st['observed_generation']==p['generation'] and st['desired_sha']==p['sha']
                and st['cluster_epoch']==self.store.cluster._read_control()['epoch'],
                '状态观测代际/epoch落后于实际配置')
        require(st['last_tick']<=tick<=st['last_tick']+self.policy.status_fresh_ticks,
                '状态过期，需重新调谐后才能放行')
        self.store.assert_aligned(p,s)
        return self.store.route(tenant,key,text,max_new_tokens)

    def manual_repair(self,lease:dict,tick:int,approval:dict) -> dict:
        self.lease.verify(lease,tick)
        st=self.read_status();require(tick>st['last_tick'],'修复tick不能倒退/重放')
        p,s=self.store.read_active(False)
        findings=self.store.detect_drift()['findings']
        require(len(findings)==1 and findings[0]['status']=='drift'
                and findings[0]['replica'] in ('A','B'),
                '人工修复只适用于一处完整可解析的副本配置漂移；其他故障另行处置')
        item=findings[0]
        required={'ticket','actor','generation','config_sha','replica','finding_sha','decision'}
        require(type(approval) is dict and set(approval)==required and
                approval['decision']=='approved' and
                approval['ticket']=='INC-0084' and
                type(approval['actor']) is str and bool(approval['actor']) and
                approval['generation']==p['generation'] and approval['config_sha']==p['sha']
                and approval['replica']==item['replica'] and
                approval['finding_sha']==digest(findings),
                '显式模拟人工审批没有绑定本次现场状态')
        self.store.cluster._check_artifacts()
        self.store.safe_capacity(s)
        repaired=self.store.reconcile(item['replica'],
                                      self.store.approval(s,p['generation']),tick,p['generation'])
        self.events.add(tick,'manual_repair',{'generation':p['generation'],
                                             'replica':item['replica'],'ticket':approval['ticket'],
                                             'field':item['path']})
        result=self.reconcile_once(tick,lease)
        require(result['ready'],'人工修复后尚未达到Ready')
        return {'repair':repaired,'status':result,'simulated_approval_only':True}

    def manual_approval(self) -> dict:
        p,_=self.store.read_active(False)
        findings=self.store.detect_drift()['findings']
        require(len(findings)==1 and findings[0].get('replica') in ('A','B'),
                '仅支持对单副本配置漂移创建人工授权夹具')
        return {'ticket':'INC-0084','actor':'local_operator','generation':p['generation'],
                'config_sha':p['sha'],'replica':findings[0]['replica'],
                'finding_sha':digest(findings),'decision':'approved'}

class SimulatedTransientRepair(RuntimeError):
    pass


def finish_request(store:ConfigStore,rid:str) -> list[dict]:
    events=[]
    for _ in range(6):
        active=store.cluster.rep[store.cluster.control['active']]
        st=active.ledger.sessions[rid]
        if st.terminal is not None:break
        events.append(store.step(rid))
    require(store.cluster.rep[store.cluster.control['active']].ledger.sessions[rid].terminal is not None,
            '请求没有在预算内到达终态')
    return events

### 先在内存示例中检查“没有写操作”的理由

`classify_drift()` 用完全可预测的字典输入；下一个完整演练用真实 `ConfigStore` 的磁盘内容驱动它，而不是人工直接指定 `ready=True`。

```text
tick1  首次安全漂移 → GracePeriod，禁止路由
 tick2 同一漂移      → GracePeriod，禁止路由
 tick3 同一漂移      → 受控自动修复，校验后恢复Ready
```

如果新增危险字段、切换 epoch 或发布 generation，不能无条件沿用之前的漂移首次观察时间与旧授权。

## 9. 端到端真实演练：六种场景

本课把真实模型推理、WAL、两个副本和调谐器放在同一条路径里执行：

1. 初始健康 + 同 tick 重复，审计事件不增加；完成请求并释放真实 KV。
2. B 副本遥测抽样漂移，宽限两 tick，拒绝请求，第三 tick 自动修复。
3. B 副本 `allowed_tenants` 漂移，不能自动覆盖；显式审批绑定现场快照才能修。
4. 上节发布事务把 `generation` 从1变为2；旧 `observed_generation` 必须阻断路由，重新观察后才能放行新流量。
5. 连续两次 I/O 错误分别退避2和4 tick，不空转、不刷新旧观察。
6. 旧 holder lease 失效后由新 holder 接管；修复写盘后模拟崩溃，新对象重启后重新观察磁盘，不能重复修复。

“自动修复权限”仅在单处低风险配置存在时启用；即使修复结束，真正放行仍需要 `assert_aligned()` 与 WAL 身份校验。

In [18]:
# %% 15 端到端演练：真实KV/WAL、幂等调谐、危险漂移、退避与重启

def make_demo(root:Path,seed:int=8401):
    tok,models=build_servable_models(seed)
    cluster=Coordinator(root/'cluster',tok,models,policy='sync',seed=seed,capacity_blocks=64)
    store=ConfigStore(root/'config',cluster)
    loop=ReconcileLoop(root/'controller',store)
    return tok,models,cluster,store,loop


def edit_replica_field(store:ConfigStore,name:str,**changes):
    p,desired=store.read_active(False)
    bad=changed_config(desired,**changes)
    atomic_json(store._replica_path(name,p['generation']),bad.wire())
    return {'replica':name,'generation':p['generation'],'changes':changes,
            'observed_sha':digest(bad.wire())}


def run_experiment(output_dir:Path=Path('lesson_84_outputs'),seed:int=8401) -> Path:
    require(type(seed) is int and seed>=0,'seed非法')
    torch.set_num_threads(1)
    root=Path(output_dir)/datetime.now(timezone.utc).strftime('run_%Y%m%d_%H%M%S_%fZ')
    root.mkdir(parents=True,exist_ok=False)
    manifest={'version':VERSION,'status':'running','seed':seed,'python':platform.python_version(),
              'torch':str(torch.__version__),'device':'cpu','sources':SOURCES,
              'boundaries':['single_process_lease_not_distributed_consensus',
                            'local_simulated_approval_not_IAM',
                            'real_cpu_KV_tensors_no_GPU_latency_claim',
                            'logical_ticks_not_walltime',
                            'safe_auto_repair_only_telemetry_sampling',
                            'no_remote_API_or_Kubernetes_cluster',
                            'local_hash_seal_not_cryptographic_authentication']}
    write_json(root/'manifest.json',manifest)
    report={}
    try:
        tok,models,cluster,store,loop=make_demo(root,seed)
        lease=loop.lease.acquire('controller_a',0,20)
        initial=loop.reconcile_once(0,lease)
        require(initial['ready'] and initial['reason']=='Ready','初始调谐失败')
        status_replay=loop.reconcile_once(0,lease)
        require(status_replay==initial,'相同tick重放必须无副作用')
        report['initial']={'generation':initial['observed_generation'],
                           'ready':initial['ready'],'idempotent_repeat':True,
                           'kv':store.probe_kv()}
        # A）低风险遥测漂移：保持请求阻断，经过2个完整逻辑tick后才修复。
        changed=edit_replica_field(store,'B',telemetry_sample_percent=80)
        safe=[]
        for tick in (1,2):
            st=loop.reconcile_once(tick,lease)
            safe.append({'tick':tick,'reason':st['reason'],'ready':st['ready']})
            require(st['reason']=='GracePeriod' and not st['ready'],'稳定性宽限内不能抢修或放行')
        try:loop.route(2,'alpha','blocked_safe','hello',4)
        except LessonError:blocked_during_grace=True
        else:raise AssertionError('漂移宽限期间不能放行')
        st=loop.reconcile_once(3,lease)
        require(st['reason']=='AutoRepaired' and st['ready'] and store.detect_drift()['healthy'],
                '安全漂移必须只修复一次')
        safe.append({'tick':3,'reason':st['reason'],'ready':st['ready']})
        req=loop.route(3,'alpha','after_auto_repair','hello',4)
        require(req['version']=='stable','自动修复不能改变模型路由')
        reply=finish_request(store,req['rid'])
        report['safe_auto']={'changed':changed,'timeline':safe,'blocked_during_grace':blocked_during_grace,
                             'model_version_after':req['version'],'generated_events':len(reply)}
        # B）权限漂移必须人工批准、绑定实际观测指纹；自动控制循环只阻断。
        unsafe=edit_replica_field(store,'B',allowed_tenants=['alpha'])
        st=loop.reconcile_once(4,lease)
        require(st['reason']=='ManualReview' and not st['ready'],'权限漂移必须人工处理')
        try:loop.route(4,'alpha','blocked_unsafe','hello',4)
        except LessonError:blocked_unsafe=True
        else:raise AssertionError('危险漂移时不能放行新流量')
        approval=loop.manual_approval()
        repaired=loop.manual_repair(lease,5,approval)
        require(repaired['status']['ready'],'显式人工审批后修复失败')
        report['manual']={'changed':unsafe,'blocked':blocked_unsafe,
                          'approval_kind':'plain_local_simulated','ready_after_manual':True,
                          'approval_finding_sha':approval['finding_sha']}
        # C）单独的发布事务升代际；调谐Status在新代际被观察之前不能继续放行。
        p0,base_cfg=store.read_active()
        candidate=changed_config(base_cfg,route_canary_percent=100,telemetry_sample_percent=20)
        proof=store.preflight(candidate,7)
        publication=store.publish(candidate,proof,store.approval(candidate,p0['generation']),
                                  p0['generation'],7)
        try:loop.route(7,'alpha','stale_generation','hello',4)
        except LessonError:stale_blocked=True
        else:raise AssertionError('Status的observedGeneration不能领先于真实观察')
        current=loop.reconcile_once(8,lease)
        require(current['observed_generation']==publication['generation'] and current['ready'],
                '新代际必须被控制循环真正观察后才Ready')
        new_req=loop.route(8,'alpha','after_publish','hello',4)
        require(new_req['version']=='canary','新配置路由没有生效')
        finish_request(store,new_req['rid'])
        report['release']={'generation':publication['generation'],
                           'stale_status_route_blocked':stale_blocked,
                           'observed_generation':current['observed_generation'],
                           'new_request_version':new_req['version']}
        # D）暂时性I/O失败：退避2、4 tick，成功后重试计数重置。
        edit_replica_field(store,'B',telemetry_sample_percent=99)
        loop.inject_remaining_failures=2
        timelines=[]
        for tick in (9,10,11,12,13,14,17):
            state=loop.reconcile_once(tick,lease)
            timelines.append({'tick':tick,'reason':state['reason'],'retry':state['retry_count'],
                              'next_retry':state['next_retry_tick'],'ready':state['ready']})
        require(timelines[2]['next_retry']==13 and timelines[4]['next_retry']==17,
                '指数退避没有遵守2、4tick')
        require(state['ready'] and state['reason']=='AutoRepaired' and state['retry_count']==0,
                '退避后应恢复，且不能保留旧失败计数')
        report['backoff']={'timeline':timelines,'max_retries':loop.policy.max_retries,
                           'succeeded_after_two_transient_failures':True}
        # E）本地lease过期后，原控制者即使持有旧token也不能写；新控制者重新取lease。
        lease_b=loop.lease.acquire('controller_b',21,70)
        try:loop.reconcile_once(21,lease)
        except LessonError:old_fenced=True
        else:raise AssertionError('过期lease必须被围栏')
        require(lease_b['term']==lease['term']+1,'租约切换必须递增term')
        state=loop.reconcile_once(21,lease_b)
        require(state['ready'],'新控制者无法观察现有期望')
        report['leader']={'old_fenced':old_fenced,'old_term':lease['term'],
                          'new_term':lease_b['term'],'no_real_distributed_leader_election':True}
        # F）修复落盘后、status落盘前模拟崩溃；重启必须看真实状态，不重放已成功修复。
        edit_replica_field(store,'B',telemetry_sample_percent=75)
        loop.reconcile_once(22,lease_b)
        loop.reconcile_once(23,lease_b)
        audit_before=store.audit.verify()['count']
        try:loop.reconcile_once(24,lease_b,crash_after_repair=True)
        except SimulatedCrash:crashed=True
        else:raise AssertionError('必须模拟after-repair-before-status崩溃')
        require(store.detect_drift()['healthy'],'崩溃点应该已经修复真实副本')
        audit_after=store.audit.verify()['count']
        require(audit_after==audit_before+1,'实际修复只写一次配置审计')
        fresh_loop=ReconcileLoop(root/'controller',store,load_existing=True)
        recovery=fresh_loop.reconcile_once(25,lease_b)
        require(recovery['ready'] and recovery['reason']=='Ready',
                '重启后必须先观察磁盘实际结果')
        require(store.audit.verify()['count']==audit_after,'重启不能重复执行修复动作')
        report['crash_recovery']={'injected':crashed,'ready':recovery['ready'],
                                  'reconcile_events':fresh_loop.events.verify()['count'],
                                  'repair_events_added_before_crash':1,
                                  'duplicate_repair_after_restart':False}
        final_req=fresh_loop.route(25,'beta','after_restart','hello',4)
        require(final_req['version']=='canary','重启后发布版本不一致')
        finish_request(store,final_req['rid'])
        # 真实的WAL / CPU KV验证：物理页在所有会话到终态后归零。
        audit=cluster.audit()
        for name in ('A','B'):
            require(cluster.rep[name].pool.audit()['live_blocks']==0,
                    '所有请求完结后真实物理KV页必须归零')
        report['final']={'ready':fresh_loop.read_status()['ready'],
                         'generation':store.read_pointer()['generation'],
                         'wal_ack_seq':cluster.control['ack_seq'],
                         'live_blocks':{name:cluster.rep[name].pool.audit()['live_blocks'] for name in ('A','B')},
                         'kv_max_abs_diff':audit['active_audit']['max_cache_full_logits_diff'],
                         'config_events':store.audit.verify()['count'],
                         'controller_events':fresh_loop.events.verify()['count'],
                         'status':fresh_loop.read_status(),
                         'physical_KV_tensor_tested':True}
        write_json(root/'experiment.json',report)
        reloaded=reload_run(root)
        write_json(root/'reload.json',reloaded)
        manifest['status']='completed';write_json(root/'manifest.json',manifest)
        print('演练：代际',report['final']['generation'],'自动修复/人工修复/退避/租约/崩溃重启均验证',flush=True)
        print('真实CPU KV回收：',report['final']['live_blocks'],flush=True)
        print('输出目录：',root,flush=True)
        return root
    except Exception as exc:
        manifest.update(status='failed',error=type(exc).__name__+': '+str(exc))
        write_json(root/'manifest.json',manifest)
        raise


def reload_run(folder:Path) -> dict:
    root=Path(folder)
    tok,models=load_model_artifacts(root/'cluster'/'models.pt')
    control=json.loads((root/'cluster'/'control.json').read_text(encoding='utf-8'))
    cluster=Coordinator(root/'cluster',tok,models,policy=control['policy'],seed=control['seed'],
                        capacity_blocks=control['capacity_blocks'],load_existing=True)
    store=ConfigStore(root/'config',cluster,load_existing=True)
    loop=ReconcileLoop(root/'controller',store,load_existing=True)
    status=loop.read_status()
    pointer,spec=store.read_active(require_healthy=True)
    require(status['observed_generation']==pointer['generation'] and
            status['desired_sha']==pointer['sha'] and status['ready'],
            '冷恢复状态没有正确观察活动版本')
    require(status['cluster_epoch']==control['epoch'],'恢复状态的epoch不一致')
    st=cluster.audit()
    for name in ('A','B'):
        require(cluster.rep[name].pool.audit()['live_blocks']==0,'冷恢复后的活动KV仍未释放')
    kv=store.probe_kv()
    return {'generation':pointer['generation'],'config_sha':pointer['sha'],
            'status_ready':status['ready'],'observed_generation':status['observed_generation'],
            'epoch':st['epoch'],'ack_seq':st['ack_seq'],
            'real_KV_vs_full_logits_max_abs_diff':kv['max_cache_full_diff'],
            'model_hash':digest(cluster.model_identity),
            'config_audit_head':store.audit.verify()['head'],
            'controller_audit_head':loop.events.verify()['head'],
            'lease_term':loop.lease.read()['term'],
            'logical_sessions':st['active_audit']['logical_requests'],
            'physical_live_KV_blocks':0,
            'single_process_no_remote_consensus':True}

In [19]:
if RUN_FULL_EXPERIMENT:
    EXPERIMENT_DIR = run_experiment(OUTPUT_ROOT,8401)
else:
    EXPERIMENT_DIR = None
    print('完整演练已关闭；不会显示虚构结果。')

演练：代际 2 自动修复/人工修复/退避/租约/崩溃重启均验证


真实CPU KV回收： {'A': 0, 'B': 0}


输出目录： lesson_84_outputs/run_20261105_082938_748292Z


## 10. 真实演练数据：不要把逻辑 tick 解释成真实时延

下面表格直接读取刚才脚本在本 Notebook 内**实际生成**的 JSON，而不是提前写好的成绩。对于 `RetryBackoff`，`next_retry_tick` 表示最早下一次允许尝试的逻辑时刻；期间控制循环仍可以检查状态并保持 NotReady。

注意：该模型随机初始化，回答的具体 token 没有任务意义；真正的数值验收是增量 KV logits 与完整前向一致，且所有终态 KV 物理页释放为零。

In [20]:
if EXPERIMENT_DIR is not None:
    run_report=json.loads((EXPERIMENT_DIR/'experiment.json').read_text(encoding='utf-8'))
    print('自动修复：',run_report['safe_auto']['timeline'])
    print('退避：')
    for entry in run_report['backoff']['timeline']:
        print(entry)
    print('危险配置已阻断：',run_report['manual']['blocked'])
    print('新generation已观察：',run_report['release']['observed_generation'])
    print('旧控制者围栏：',run_report['leader']['old_fenced'])
    print('修复后崩溃重启未重复修复：',not run_report['crash_recovery']['duplicate_repair_after_restart'])
    print('最终真实KV页：',run_report['final']['live_blocks'])

自动修复： [{'ready': False, 'reason': 'GracePeriod', 'tick': 1}, {'ready': False, 'reason': 'GracePeriod', 'tick': 2}, {'ready': True, 'reason': 'AutoRepaired', 'tick': 3}]
退避：
{'next_retry': 0, 'ready': False, 'reason': 'GracePeriod', 'retry': 0, 'tick': 9}
{'next_retry': 0, 'ready': False, 'reason': 'GracePeriod', 'retry': 0, 'tick': 10}
{'next_retry': 13, 'ready': False, 'reason': 'RetryBackoff', 'retry': 1, 'tick': 11}
{'next_retry': 13, 'ready': False, 'reason': 'RetryBackoff', 'retry': 1, 'tick': 12}
{'next_retry': 17, 'ready': False, 'reason': 'RetryBackoff', 'retry': 2, 'tick': 13}
{'next_retry': 17, 'ready': False, 'reason': 'RetryBackoff', 'retry': 2, 'tick': 14}
{'next_retry': 0, 'ready': True, 'reason': 'AutoRepaired', 'retry': 0, 'tick': 17}
危险配置已阻断： True
新generation已观察： 2
旧控制者围栏： True
修复后崩溃重启未重复修复： True
最终真实KV页： {'A': 0, 'B': 0}


## 11. 崩溃时的正确幂等：重新观察，而不是重复发命令

故障发生在以下位置：

```text
B副本文件已被受控修复
    ↓   ← 注入崩溃
status.json 还没有记为Ready
```

控制器重新构造对象后发现：两副本**已经一致**。于是只将 Status 更新为 Ready；**不再次调用修复，也不生成第二次修复审计**。

这类幂等性来自“按期望/实际状态重算动作”，不是记录了一个 `already_done` 布尔值就不再做任何检查。反过来，如果写修复失败，持久实际仍未改变，后续可以在明确定义的退避内重试。

本课审计哈希链的本地 head、状态封印和本地 lease 都是教学完整性保护，并非经外部锚定的不可伪造审计。生产环境需要持久控制平面、真实字段所有权、权限与并发事务。

## 12. 冷重载：只验证可信本地状态与实际 KV

`reload_run()` 在新的对象图中重新加载磁盘工件和 WAL：重新核对活动指针、Status 的代际与 SHA、模型身份、控制 epoch、变更与调谐审计、物理 KV 释放及独立 KV 数值探针。

它不会将旧请求任意重新排队，也不会复原真实 GPU 缓存页。这里只从 WAL 重建必要逻辑，再走本地 CPU 计算路径；**不是远程灾难恢复产品**。

In [21]:
if EXPERIMENT_DIR is not None:
    reloaded=reload_run(EXPERIMENT_DIR)
    print(json.dumps(reloaded,ensure_ascii=False,indent=2))

{
  "generation": 2,
  "config_sha": "19168ec5a64cb14af67e8911ceb476c25b06ca81174c3308bf032015745a9ff4",
  "status_ready": true,
  "observed_generation": 2,
  "epoch": 1,
  "ack_seq": 15,
  "real_KV_vs_full_logits_max_abs_diff": 3.5762786865234375e-07,
  "model_hash": "a8d4919487ffa569a627138e5dd82ef99c5c3749cc2113c02db9b0600f95f4a2",
  "config_audit_head": "d31d74454d8dc5ef08d5c492074177b1f644a053dff75167fccba5c10d69467d",
  "controller_audit_head": "3e59a624158d01ddca4f6dd13e3f5cdb4116b2e44be5780efe17e6b83bb5ebb7",
  "lease_term": 2,
  "logical_sessions": 3,
  "physical_live_KV_blocks": 0,
  "single_process_no_remote_consensus": true
}


## 13. 固定检查：会拒绝的系统，比“看起来正常”更有意义

自动检查包含：字段白名单、健康/危险分类、租约过期、同 tick 重放、宽限期阻断、人工审批与现场绑定、发布代际失效、退避、最大失败隔离、旧状态失效、损坏状态封印、损坏配置、模型数值路径和冷恢复。

这份检查表不会声称已经执行真实外部并发、真实Kubernetes API、跨节点共识或GPU稳定性测试。它验证**本课的确定性教学契约**。

In [22]:
# %% 16 可执行拒绝检查、反例与故障恢复验收
class Checks:
    def __init__(self):self.names=[]
    def ok(self,name,value):
        if not bool(value):raise AssertionError(name)
        self.names.append(name)
    def equal(self,name,actual,expected):self.ok(name,actual==expected)
    def close(self,name,a,b,tol=2e-5):self.ok(name,abs(a-b)<=tol)
    def rejects(self,name,fn):
        try:fn()
        except (LessonError,ValueError):self.names.append(name);return
        raise AssertionError(name+': 应拒绝却未拒绝')


def run_checks() -> dict:
    torch.set_num_threads(1)
    c=Checks();policy=ReconcilePolicy();policy.validate()
    c.equal('唯一自动修复字段',policy.auto_paths,('telemetry_sample_percent',))
    c.equal('默认观察宽限',policy.grace_ticks,2)
    c.equal('默认退避基础',policy.retry_base_ticks,2)
    c.equal('默认退避上限',policy.retry_cap_ticks,8)
    c.equal('默认最多三次失败',policy.max_retries,3)
    c.rejects('危险路径不能被列为自动修复白名单',lambda:ReconcilePolicy(auto_paths=('allowed_tenants',)).validate())
    c.rejects('额外路由字段不能偷偷加入白名单',lambda:ReconcilePolicy(auto_paths=('telemetry_sample_percent','route_canary_percent')).validate())
    c.rejects('空白白名单拒绝',lambda:ReconcilePolicy(auto_paths=()).validate())
    c.rejects('非法退避参数拒绝',lambda:ReconcilePolicy(retry_base_ticks=0).validate())
    c.rejects('最大退避小于基础拒绝',lambda:ReconcilePolicy(retry_base_ticks=8,retry_cap_ticks=2).validate())
    c.rejects('非法重试次数拒绝',lambda:ReconcilePolicy(max_retries=-1).validate())
    c.rejects('非整型宽限拒绝',lambda:ReconcilePolicy(grace_ticks=True).validate())
    low={'replica':'B','path':'telemetry_sample_percent','status':'drift','severity':'controlled'}
    high={'replica':'B','path':'allowed_tenants','status':'drift','severity':'critical'}
    route={'replica':'B','path':'route_canary_percent','status':'drift','severity':'controlled'}
    invalid={'replica':'B','status':'invalid','reason':'corrupt'}
    c.equal('空漂移健康',classify_drift([],policy)['mode'],'healthy')
    c.equal('单处遥测漂移可受控修复',classify_drift([low],policy)['mode'],'auto_safe')
    c.equal('安全修复只能指向实际副本',classify_drift([low],policy)['replica'],'B')
    for name,items in [('路由变更',[route]),('权限变更',[high]),('未知配置字段',[{**low,'path':'unknown'}]),
                       ('权限与遥测混合',[low,high]),('双副本遥测',[low,{**low,'replica':'A'}]),
                       ('损坏配置',[invalid]),('模型实际权重变化',[{'replica':'models','status':'drift','path':'model_identity_hash'}]),
                       ('WAL模式漂移',[{'replica':'wal','status':'drift','path':'wal_policy'}])]:
        c.equal(name+'必须人工处理',classify_drift(items,policy)['mode'],'manual')
    c.rejects('分类器不能接收字典代替列表',lambda:classify_drift({'x':1},policy))
    c.rejects('非法漂移记录拒绝',lambda:classify_drift(['drift'],policy))
    with tempfile.TemporaryDirectory() as tmp:
        root=Path(tmp)/'fixture'
        tok,models,cluster,store,loop=make_demo(root,8402)
        lease=loop.lease.acquire('operator_a',0,16)
        status=loop.reconcile_once(0,lease)
        c.ok('初始状态Ready',status['ready'])
        c.equal('初始实际代际',status['observed_generation'],1)
        c.equal('初始原因Ready',status['reason'],'Ready')
        c.ok('liveness与readiness分开',status['live'] and status['ready'])
        initial_audit=loop.events.verify()['count']
        c.equal('同tick重复调谐返回相同状态',loop.reconcile_once(0,lease),status)
        c.equal('同tick重放不重复写审计',loop.events.verify()['count'],initial_audit)
        c.rejects('拒绝倒退逻辑时钟',lambda:loop.reconcile_once(-1,lease))
        c.rejects('过期的ready状态不能路由',lambda:loop.route(3,'alpha','stale','hello',4))
        valid=loop.route(0,'alpha','open','hello',4)
        c.ok('真实WAL接收请求',valid['new_event'])
        c.equal('初始请求绑定stable',valid['version'],'stable')
        ended=finish_request(store,valid['rid'])
        c.ok('真实生成达到终态',len(ended)>0)
        c.equal('请求终态物理KV归零',cluster.rep['A'].pool.audit()['live_blocks'],0)
        c.ok('实际缓存logits符合全前向',store.probe_kv()['max_cache_full_diff']<2e-5)
        c.equal('lease同持有者重入幂等',loop.lease.acquire('operator_a',0,16),lease)
        c.rejects('活跃lease拒绝另一持有者',lambda:loop.lease.acquire('operator_b',1,12))
        c.rejects('lease输入不允许非法字符',lambda:loop.lease.acquire('bad holder',1,12))
        wrong_lease=copy.deepcopy(lease);wrong_lease['term']+=1
        c.rejects('伪造term不能持有lease',lambda:loop.reconcile_once(1,wrong_lease))
        # 低风险漂移先经过观察宽限，再修复；检查审计只添加一次。
        edit_replica_field(store,'B',telemetry_sample_percent=90)
        drift_report=store.detect_drift()
        c.ok('实际检测B副本配置漂移',not drift_report['healthy'])
        for tick in (1,2):
            g=loop.reconcile_once(tick,lease)
            c.equal(f'宽限{tick}不自动修复',g['reason'],'GracePeriod')
            c.ok(f'宽限{tick}保持NotReady',not g['ready'])
            c.rejects(f'宽限{tick}拒绝流量',lambda tick=tick:loop.route(tick,'alpha','blocked'+str(tick),'hello',4))
        before=store.audit.verify()['count']
        fixed=loop.reconcile_once(3,lease)
        c.equal('宽限结束自动修复',fixed['reason'],'AutoRepaired')
        c.ok('恢复为Ready',fixed['ready'])
        c.equal('一次自动修复只写一个配置审计',store.audit.verify()['count'],before+1)
        c.ok('修复后实际全部副本一致',store.detect_drift()['healthy'])
        c.equal('相同tick自动修复重放无二次操作',loop.reconcile_once(3,lease),fixed)
        c.equal('重放不重复修复审计',store.audit.verify()['count'],before+1)
        # 危险漂移：同一控制者也不能静默覆盖权限。
        edit_replica_field(store,'B',allowed_tenants=['alpha'])
        manual=loop.reconcile_once(4,lease)
        c.equal('权限漂移进入人工状态',manual['reason'],'ManualReview')
        c.ok('危险状态拒绝流量但保持存活',manual['live'] and not manual['ready'])
        c.rejects('危险漂移拒绝新路由',lambda:loop.route(4,'alpha','new_manual','hello',4))
        wrong=loop.manual_approval();wrong['ticket']='OTHER'
        c.rejects('错误工单不能授权修复',lambda:loop.manual_repair(lease,5,wrong))
        wrong=loop.manual_approval();wrong['finding_sha']='0'*64
        c.rejects('观察现场改变后的审批不能复用',lambda:loop.manual_repair(lease,5,wrong))
        good=loop.manual_approval()
        actual=loop.manual_repair(lease,5,good)
        c.ok('人工明确修复后状态Ready',actual['status']['ready'])
        c.ok('人工事件链含人工记录','manual_repair' in loop.events.verify()['events'])
        c.rejects('不能在健康状态重复人工修复',lambda:loop.manual_repair(lease,6,good))
        # 无权修改已发布generation：先走独立事务，再重新观察。
        p,desired=store.read_active()
        candidate=changed_config(desired,telemetry_sample_percent=30)
        preflight=store.preflight(candidate,6)
        published=store.publish(candidate,preflight,store.approval(candidate,1),1,6)
        c.equal('发布事务增加generation',published['generation'],2)
        c.rejects('发布后旧状态必须阻断路由',lambda:loop.route(6,'alpha','stale_gen','hello',4))
        ready=loop.reconcile_once(7,lease)
        c.equal('status真实观察新generation',ready['observed_generation'],2)
        c.ok('新generationReady',ready['ready'])
        c.equal('status绑定真实活动sha',ready['desired_sha'],store.read_pointer()['sha'])
        # 临时错误：退避2/4tick；下一次才修复。
        edit_replica_field(store,'B',telemetry_sample_percent=99)
        loop.inject_remaining_failures=2
        loop.reconcile_once(8,lease)
        loop.reconcile_once(9,lease)
        f1=loop.reconcile_once(10,lease)
        c.equal('第一次失败延迟两tick',f1['next_retry_tick'],12)
        c.equal('第一次失败只记一次',f1['retry_count'],1)
        wait=loop.reconcile_once(11,lease)
        c.equal('退避期间不重试',wait['retry_count'],1)
        f2=loop.reconcile_once(12,lease)
        c.equal('第二次失败延迟四tick',f2['next_retry_tick'],16)
        c.equal('第二次失败计数',f2['retry_count'],2)
        healed=loop.reconcile_once(16,lease)
        c.ok('指数退避后最终Ready',healed['ready'])
        c.equal('成功后失败计数清零',healed['retry_count'],0)
        # 新控制者取过期lease，旧token不可用。
        new_lease=loop.lease.acquire('operator_b',17,40)
        c.equal('lease持有者变化term递增',new_lease['term'],lease['term']+1)
        c.rejects('旧持有者被围栏',lambda:loop.reconcile_once(17,lease))
        c.ok('新持有者可以调谐',loop.reconcile_once(17,new_lease)['ready'])
        # 有效但即将过期的状态不能绕过检测；后继心跳持续刷新。
        c.rejects('过期的调谐状态拒绝流量',lambda:loop.route(20,'alpha','stale2','hello',4))
        c.ok('新心跳恢复Ready',loop.reconcile_once(20,new_lease)['ready'])
        # 状态封印篡改：不能仅因为readiness为True就放行。
        saved=loop.status_path.read_bytes();raw=json.loads(saved)
        raw['status']['desired_sha']='fake'
        atomic_json(loop.status_path,raw)
        c.rejects('状态被改但摘要未更新则拒绝',lambda:loop.read_status())
        c.rejects('损坏状态不能用于新请求',lambda:loop.route(20,'alpha','must_block','hello',4))
        loop.status_path.write_bytes(saved)
        c.ok('恢复本地可信状态后可继续读取',loop.read_status()['ready'])
        # 模拟再次故障并达到最大失败次数，需人工处置，不盲目重试。
        edit_replica_field(store,'B',telemetry_sample_percent=84)
        loop.inject_remaining_failures=3
        loop.reconcile_once(21,new_lease)
        loop.reconcile_once(22,new_lease)
        a=loop.reconcile_once(23,new_lease)
        b=loop.reconcile_once(25,new_lease)
        q=loop.reconcile_once(29,new_lease)
        c.equal('第三次失败进入隔离',q['reason'],'Quarantined')
        c.ok('隔离状态NotReady',q['quarantined'] and not q['ready'])
        c.equal('达到最大失败次数后停止计划重试',q['next_retry_tick'],0)
        c.equal('隔离时真实漂移仍在',store.detect_drift()['healthy'],False)
        c.rejects('隔离后仍拒绝新请求',lambda:loop.route(29,'alpha','quarantined','hello',4))
        ongoing=loop.reconcile_once(30,new_lease)
        c.equal('原漂移没变时保持隔离',ongoing['reason'],'Quarantined')
        approval=loop.manual_approval()
        recovered=loop.manual_repair(new_lease,31,approval)
        c.ok('显式人工授权可恢复隔离状态',recovered['status']['ready'])
        # 重启状态和审计可被独立对象读取（不复用old reference）。
        restarted=ReconcileLoop(root/'controller',store,load_existing=True)
        c.equal('重启保持最后状态',restarted.read_status(),loop.read_status())
        c.ok('重启能核对审计头',restarted.events.verify()['count']>=1)
        c.ok('真正CPU KV仍可验证',store.probe_kv()['max_cache_full_diff']<2e-5)
        c.ok('本机两副本的KV页归零',all(cluster.rep[n].pool.audit()['live_blocks']==0 for n in ('A','B')))
        # 文件损坏的存储级故障，不能退化成自动修复未知字段。
        spec_file=store._replica_path('B',store.read_pointer()['generation'])
        clean=spec_file.read_bytes();spec_file.write_text('{broken',encoding='utf-8')
        report=store.detect_drift()
        c.equal('损坏配置被标记invalid',report['findings'][0]['status'],'invalid')
        bad_status=restarted.reconcile_once(32,new_lease)
        c.equal('损坏副本必须人工处理',bad_status['reason'],'ManualReview')
        c.rejects('损坏配置不进行自动覆盖',lambda:restarted.route(32,'alpha','invalid','hello',4))
        spec_file.write_bytes(clean)
        c.ok('损坏副本恢复后重新调谐才能Ready',restarted.reconcile_once(33,new_lease)['ready'])
    # 与其他字段完全相同，但发布指针的代际已经变化；不能仅用SHA绕过observedGeneration。
    with tempfile.TemporaryDirectory() as tmp:
        root=Path(tmp)/'identical_generation'
        _tok,_models,_cluster,store,loop=make_demo(root,8403)
        lease=loop.lease.acquire('operator',0,10)
        old=loop.reconcile_once(0,lease)
        prev_pointer,same_config=store.read_active()
        proof=store.preflight(same_config,1)
        published=store.publish(same_config,proof,store.approval(same_config,1),1,1)
        pointer,_=store.read_active()
        c.equal('相同SHA也能发布新generation',published['generation'],2)
        c.equal('相同内容的不同generation共享sha',pointer['sha'],old['desired_sha'])
        c.rejects('即便SHA相同旧observedGeneration也不能放行',
                  lambda:loop.route(1,'alpha','same_sha_gen','hello',4))
        fresh=loop.reconcile_once(2,lease)
        c.equal('新代际必须确实被观察',fresh['observed_generation'],2)
        c.ok('重调谐后才恢复ready',fresh['ready'])
    with tempfile.TemporaryDirectory() as tmp:
        result=run_experiment(Path(tmp),8401)
        data=json.loads((result/'experiment.json').read_text(encoding='utf-8'))
        reread=reload_run(result)
        c.ok('独立演练包含真正KV探针',data['initial']['kv']['real_tensor_kv'])
        c.ok('自动安全修复经过两tick宽限',len(data['safe_auto']['timeline'])==3)
        c.ok('危险漂移必须阻断路由',data['manual']['blocked'])
        c.ok('发布后旧observedGeneration阻断路由',data['release']['stale_status_route_blocked'])
        c.ok('两次模拟I/O错误后退避恢复',data['backoff']['succeeded_after_two_transient_failures'])
        c.ok('过期旧leader被拒绝',data['leader']['old_fenced'])
        c.ok('崩溃后没有重复执行已落盘修复',not data['crash_recovery']['duplicate_repair_after_restart'])
        c.ok('实际KV块均已释放',set(data['final']['live_blocks'].values())=={0})
        c.equal('持久代际与冷恢复一致',data['final']['generation'],reread['generation'])
        c.ok('冷恢复状态Ready',reread['status_ready'])
        c.ok('冷恢复后KV与完整前向一致',reread['real_KV_vs_full_logits_max_abs_diff']<2e-5)
        c.equal('冷恢复epoch一致',reread['epoch'],1)
    return {'passed':len(c.names),'names':c.names,'scope':'本地单进程双副本真实CPU KV、WAL与逻辑tick控制循环；无真实集群共识'}

In [23]:
check_report=run_checks()
print('固定检查通过：',check_report['passed'],'项')
print('适用范围：',check_report['scope'])
if EXPERIMENT_DIR is not None:
    write_json(EXPERIMENT_DIR/'checks.json',check_report)

演练：代际 2 自动修复/人工修复/退避/租约/崩溃重启均验证


真实CPU KV回收： {'A': 0, 'B': 0}


输出目录： /tmp/tmpgvx9jjqr/run_20261105_082940_478492Z


固定检查通过： 115 项
适用范围： 本地单进程双副本真实CPU KV、WAL与逻辑tick控制循环；无真实集群共识


## 14. 独立练习A：只允许单处遥测漂移自动修

实现 `classify_ex(findings)`，返回 `'healthy'`、`'auto_safe'` 或 `'manual'`：

- 空 findings 健康。
- 只有一处合法的 `B.telemetry_sample_percent`、`status='drift'`、`severity='controlled'` 可自动修（也可允许 A）。
- 权限、未知字段、损坏记录和多处漂移一律人工。
- 注意**多条相同记录也不应直接当作一条**。

先亲手补写，默认检查关闭。

In [24]:
def exercise_check_a(fn:Callable):
    low={'replica':'B','status':'drift','path':'telemetry_sample_percent','severity':'controlled'}
    other={**low,'path':'allowed_tenants','severity':'critical'}
    assert fn([])=='healthy'
    assert fn([low])=='auto_safe'
    assert fn([other])=='manual'
    assert fn([low,low])=='manual'
    assert fn([{'replica':'A','status':'invalid'}])=='manual'
    print('练习A：单处白名单与危险漂移分类通过')
    return {'exercise':'A','passed':True}

def exercise_check_b(fn:Callable):
    assert [fn(i,2,8) for i in range(1,6)]==[2,4,8,8,8]
    for args in [(0,2,8),(1,0,8),(2,8,4)]:
        try:fn(*args)
        except ValueError:pass
        else:raise AssertionError('退避参数必须拒绝')
    print('练习B：指数退避与上限通过')
    return {'exercise':'B','passed':True}

def exercise_check_c(fn:Callable):
    good={'ready':True,'live':True,'quarantined':False,'observed_generation':3,
          'desired_sha':'abc','cluster_epoch':7,'last_tick':10}
    assert fn(good,3,'abc',7,11,2)
    for patch in ({'ready':False},{'live':False},{'quarantined':True},
                  {'observed_generation':2},{'desired_sha':'x'},
                  {'cluster_epoch':8},{'last_tick':8}):
        altered={**good,**patch}
        try:fn(altered,3,'abc',7,11,2)
        except ValueError:pass
        else:raise AssertionError('应该拒绝过期/不一致的Status')
    print('练习C：Status与真实代际、epoch、心跳绑定通过')
    return {'exercise':'C','passed':True}

In [25]:
def classify_ex(findings):
    raise NotImplementedError('请实现安全分类')

RUN_EXERCISE_A=False
if RUN_EXERCISE_A:
    exercise_check_a(classify_ex)

## 15. 独立练习B：指数退避与上限

实现 `backoff_ex(attempt,base,cap)`：第一次失败等待 base tick；随后翻倍，不超过 cap。验证参数合法，拒绝 `attempt=0`、`base=0`、`cap<base`。

例如 `(base=2,cap=8)`，第1到第5次的结果是 `2,4,8,8,8`。**不要把它当作睡眠秒数或墙钟计时。**

In [26]:
def backoff_ex(attempt,base,cap):
    raise NotImplementedError('请实现退避tick')

RUN_EXERCISE_B=False
if RUN_EXERCISE_B:
    exercise_check_b(backoff_ex)

## 16. 独立练习C：用 Status 做路由门控

实现 `status_guard_ex(status,generation,sha,epoch,tick,ttl)`：

- `ready=True`、`live=True`、`quarantined=False`。
- `observed_generation`、`desired_sha`、`cluster_epoch` 必须与当前真实控制上下文相同。
- `last_tick <= tick <= last_tick + ttl`。
- 不满足时抛 `ValueError` 或其子类；健康返回 `True`。

它是纯函数练习。完整服务还要继续核对真实副本配置、租户、容量、WAL与模型身份，不能只靠这个状态对象路由。

In [27]:
def status_guard_ex(status,generation,sha,epoch,tick,ttl):
    raise NotImplementedError('请实现Status门控')

RUN_EXERCISE_C=False
if RUN_EXERCISE_C:
    exercise_check_c(status_guard_ex)

## 17. 参考答案

先完成自己的三个实现，再展开对照。参考答案采用 `answer_a / answer_b / answer_c`，不会覆盖你的函数。

In [28]:
def answer_a(findings:list[dict]) -> str:
    return classify_drift(findings,ReconcilePolicy())['mode']

def answer_b(attempt:int,base:int,cap:int) -> int:
    if (type(attempt) is not int or type(base) is not int or type(cap) is not int
            or attempt<1 or base<1 or cap<base):
        raise ValueError('需要合法的尝试序号、基础tick和上限')
    return min(cap,base*(2**(attempt-1)))

def answer_c(status:dict,generation:int,sha:str,epoch:int,tick:int,ttl:int) -> bool:
    if (type(tick) is not int or type(ttl) is not int or tick<0 or ttl<1 or
        status.get('ready') is not True or status.get('live') is not True or
        status.get('quarantined') is not False or
        status.get('observed_generation')!=generation or status.get('desired_sha')!=sha or
        status.get('cluster_epoch')!=epoch or
        type(status.get('last_tick')) is not int or
        not status['last_tick']<=tick<=status['last_tick']+ttl):
        raise LessonError('Status不可用于当前路由')
    return True

In [29]:
exercise_check_a(answer_a)
exercise_check_b(answer_b)
exercise_check_c(answer_c)

练习A：单处白名单与危险漂移分类通过
练习B：指数退避与上限通过
练习C：Status与真实代际、epoch、心跳绑定通过


{'exercise': 'C', 'passed': True}

## 18. 独立脚本、局限与下一课

```bash
python lesson_84_declarative_reconcile_safety.py --checks-only
python lesson_84_declarative_reconcile_safety.py --math-only
python lesson_84_declarative_reconcile_safety.py
python lesson_84_declarative_reconcile_safety.py --reload-run lesson_84_outputs/实际运行目录
```

本课代码只在本地单进程序列化执行，没有 K8s API、LeaderElection、真实 IAM、多进程 CAS 或外部一致性存储。两份文件完全自包含。CPU 真实 KV logits 计算正确，不代表对真实集群安全性、吞吐量或GPU显存的验证。

**下一课建议：第85课——控制平面故障状态机、事件重放与工作队列背压。** 继续用受控模拟检查乱序、重复事件、延迟重试和调谐幂等，而不是让控制循环空转或静默放大故障。

### 资料

- [Kubernetes：Controllers](https://kubernetes.io/docs/concepts/architecture/controller/)
- [Kubernetes：Server-Side Apply字段所有权/冲突](https://kubernetes.io/docs/reference/using-api/server-side-apply/)
- [Red Hat：Operator错误管理与退避](https://www.redhat.com/en/blog/kubernetes-operators-best-practices)
- [NIST：安全配置管理基线与变更控制](https://csrc.nist.gov/pubs/sp/800/128/upd1/final)
- [PyTorch复现边界](https://docs.pytorch.org/docs/stable/notes/randomness.html)